# Transformer - Notebook Autonome (Rendu)

Ce notebook est **autonome**: le code du pipeline Transformer est embarqué dans les cellules
et exécuté directement dans le notebook (sans appel `!python train.py` / `!python evaluate.py`).

Note importante: ce workflow a été principalement développé et validé en local Windows/PowerShell.


## Cellule 1 - Installation (si besoin)

Installe les dépendances minimales. Ignore cette cellule si ton environnement est déjà prêt.


In [ ]:
# %pip install -q numpy pandas matplotlib scikit-learn torch tqdm


## Cellule 2 - Chemins projet/données

Adapte ces chemins selon ton environnement local ou Colab/Drive.


In [ ]:
from pathlib import Path

# Repertoire racine du projet
PROJECT_ROOT = Path.cwd()

# Donnees (option recommande: NPZ prepare)
PREPARED_NPZ = PROJECT_ROOT / "data" / "parcel_dataset_ext.npz"
CSV_PATH = PROJECT_ROOT / "data" / "s2_herault_2024_full" / "indices_parcelles_2024-01-01_2024-12-31_win5d_with_labels_and_group_min200_ext.csv"

# Dossier de sortie pour ce notebook
OUTPUT_ROOT = PROJECT_ROOT / "outputs_transformer_notebook"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PREPARED_NPZ:", PREPARED_NPZ)
print("CSV_PATH:", CSV_PATH)
print("OUTPUT_ROOT:", OUTPUT_ROOT)


## Cellule 3 - Chargement du pipeline embarqué

Cette cellule charge le code des modules (`config`, `data`, `model`, `evaluate`, `pretrain_ssl`, `train`, etc.)
directement en mémoire du notebook.


In [ ]:
import sys
import types

embedded_sources = {}
embedded_sources['config'] = 'from __future__ import annotations\n\nfrom dataclasses import asdict, dataclass, field\nfrom typing import Literal, Optional\n\nSplitMethod = Literal["parcel", "tile"]\nPoolingType = Literal["cls", "mean"]\nSchedulerType = Literal["none", "plateau", "cosine"]\nLossType = Literal["cross_entropy", "focal", "balanced_softmax", "logit_adjusted"]\n\nDEFAULT_INDEX_FILTER = [\n    "NDVI",\n    "NDMI",\n    "NDWI",\n    "GNDVI",\n    "SAVI",\n    "OSAVI",\n    "MSAVI",\n    "MNDWI",\n    "ARVI",\n    "BSI",\n]\n\n\n@dataclass\nclass DataConfig:\n    csv_path: str = (\n        "data/s2_herault_2024_full_year_5day_cloudmask_fast/"\n        "indices_parcelles_2024-01-01_2024-12-31_win5d_with_labels.csv"\n    )\n    output_dir: str = "outputs_transformer"\n    parcel_id_col: str = "ID_PARCEL"\n    date_col: str = "date"\n    index_col: str = "index"\n    value_col: str = "value_mean"\n    label_col: str = "label"\n    label_group_col: str = "label_group"\n    tile_col: str = "tile"\n    cloud_col: str = "cloud_scene"\n    px_count_col: str = "px_count"\n    index_filter: list[str] = field(default_factory=lambda: list(DEFAULT_INDEX_FILTER))\n    min_px_count: int = 0\n    max_cloud_scene: Optional[float] = None\n    min_obs_per_parcel: int = 4\n    fill_value: float = 0.0\n    time_grid_frequency: Optional[str] = None\n    split_method: SplitMethod = "parcel"\n    stratify: bool = True\n    test_size: float = 0.2\n    val_size: float = 0.1\n    random_state: int = 42\n    prepared_npz_path: Optional[str] = None\n    save_prepared_npz_path: Optional[str] = None\n\n\n@dataclass\nclass ModelConfig:\n    d_model: int = 128\n    n_heads: int = 4\n    n_layers: int = 3\n    dim_feedforward: int = 256\n    dropout: float = 0.1\n    pooling: PoolingType = "cls"\n    use_layer_norm_first: bool = True\n    reliability_aware: bool = False\n\n\n@dataclass\nclass TrainConfig:\n    epochs: int = 80\n    batch_size: int = 64\n    learning_rate: float = 1e-3\n    weight_decay: float = 1e-2\n    standardize_features: bool = True\n    standardize_eps: float = 1e-6\n    loss_type: LossType = "focal"\n    focal_gamma: float = 2.0\n    logit_adjust_tau: float = 1.0\n    use_group_task: bool = False\n    group_loss_weight: float = 0.3\n    hierarchical_constraint: bool = False\n    hierarchical_constraint_weight: float = 1.0\n    hierarchical_constraint_eps: float = 1e-6\n    class_weighting: bool = True\n    class_weight_power: float = 0.5\n    weighted_sampler: bool = False\n    sampler_power: float = 1.0\n    temporal_augmentation: bool = False\n    time_mask_ratio: float = 0.0\n    jitter_std: float = 0.0\n    phase2_rare_finetune: bool = False\n    phase2_epochs: int = 12\n    phase2_learning_rate: float = 1e-4\n    phase2_sampler_power: float = 1.0\n    phase2_rare_quantile: float = 0.25\n    phase2_rare_count_threshold: Optional[int] = None\n    phase2_rare_boost: float = 2.0\n    phase2_early_stopping_patience: int = 6\n    scheduler: SchedulerType = "plateau"\n    scheduler_patience: int = 5\n    scheduler_factor: float = 0.5\n    min_learning_rate: float = 1e-6\n    early_stopping_patience: int = 12\n    gradient_clip_norm: Optional[float] = 1.0\n    num_workers: int = 0\n    seed: int = 42\n    device: str = "auto"\n\n\n@dataclass\nclass ExperimentConfig:\n    data: DataConfig = field(default_factory=DataConfig)\n    model: ModelConfig = field(default_factory=ModelConfig)\n    train: TrainConfig = field(default_factory=TrainConfig)\n\n    def to_dict(self) -> dict:\n        return asdict(self)\n\n\ndef get_default_config() -> ExperimentConfig:\n    return ExperimentConfig()\n'
embedded_sources['utils'] = 'from __future__ import annotations\n\nimport json\nimport logging\nimport random\nfrom datetime import datetime\nfrom pathlib import Path\nfrom typing import Any, Optional\n\nimport numpy as np\nimport torch\n\n\ndef set_seed(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False\n\n\ndef resolve_device(device_arg: str = "auto") -> torch.device:\n    if device_arg == "auto":\n        return torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    return torch.device(device_arg)\n\n\ndef create_run_dir(base_dir: str | Path, prefix: str = "run") -> Path:\n    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")\n    run_dir = Path(base_dir) / f"{prefix}_{timestamp}"\n    run_dir.mkdir(parents=True, exist_ok=True)\n    return run_dir\n\n\ndef setup_logger(log_file: str | Path, level: int = logging.INFO) -> logging.Logger:\n    logger = logging.getLogger("parcel_transformer")\n    logger.setLevel(level)\n    logger.handlers.clear()\n\n    formatter = logging.Formatter(\n        fmt="%(asctime)s | %(levelname)s | %(name)s | %(message)s",\n        datefmt="%Y-%m-%d %H:%M:%S",\n    )\n\n    stream_handler = logging.StreamHandler()\n    stream_handler.setFormatter(formatter)\n    logger.addHandler(stream_handler)\n\n    file_handler = logging.FileHandler(log_file, encoding="utf-8")\n    file_handler.setFormatter(formatter)\n    logger.addHandler(file_handler)\n    logger.propagate = False\n    return logger\n\n\ndef save_json(data: dict[str, Any], path: str | Path) -> None:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", encoding="utf-8") as f:\n        json.dump(data, f, indent=2, ensure_ascii=True)\n\n\ndef save_checkpoint(\n    path: str | Path,\n    model: torch.nn.Module,\n    optimizer: torch.optim.Optimizer,\n    scheduler: Optional[torch.optim.lr_scheduler.LRScheduler],\n    epoch: int,\n    best_metric: float,\n    metadata: Optional[dict[str, Any]] = None,\n) -> None:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n\n    checkpoint = {\n        "model_state_dict": model.state_dict(),\n        "optimizer_state_dict": optimizer.state_dict(),\n        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,\n        "epoch": epoch,\n        "best_metric": best_metric,\n        "metadata": metadata or {},\n    }\n    torch.save(checkpoint, path)\n\n\ndef load_checkpoint(\n    path: str | Path,\n    model: torch.nn.Module,\n    optimizer: Optional[torch.optim.Optimizer] = None,\n    scheduler: Optional[torch.optim.lr_scheduler.LRScheduler] = None,\n    map_location: str | torch.device = "cpu",\n) -> dict[str, Any]:\n    path = Path(path)\n    if not path.exists():\n        raise FileNotFoundError(f"Checkpoint not found: {path}")\n\n    try:\n        checkpoint = torch.load(path, map_location=map_location, weights_only=False)\n    except TypeError:\n        checkpoint = torch.load(path, map_location=map_location)\n    model.load_state_dict(checkpoint["model_state_dict"])\n\n    if optimizer is not None and checkpoint.get("optimizer_state_dict") is not None:\n        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])\n    if scheduler is not None and checkpoint.get("scheduler_state_dict") is not None:\n        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])\n    return checkpoint\n\n\ndef compute_class_weights(\n    labels: np.ndarray,\n    num_classes: int,\n    power: float = 1.0,\n    min_weight: Optional[float] = None,\n    max_weight: Optional[float] = None,\n) -> np.ndarray:\n    counts = np.bincount(labels, minlength=num_classes).astype(np.float64)\n    if np.any(counts == 0):\n        counts = np.maximum(counts, 1.0)\n    weights = counts.sum() / (num_classes * counts)\n    if power != 1.0:\n        weights = np.power(weights, power)\n    if min_weight is not None:\n        weights = np.maximum(weights, float(min_weight))\n    if max_weight is not None:\n        weights = np.minimum(weights, float(max_weight))\n    weights = weights / np.clip(weights.mean(), a_min=1e-12, a_max=None)\n    return weights.astype(np.float32)\n\n\nclass EarlyStopping:\n    def __init__(self, patience: int, mode: str = "max", min_delta: float = 1e-4) -> None:\n        if mode not in {"min", "max"}:\n            raise ValueError("mode must be \'min\' or \'max\'.")\n        self.patience = patience\n        self.mode = mode\n        self.min_delta = min_delta\n        self.best_score: Optional[float] = None\n        self.counter = 0\n\n    def step(self, score: float) -> bool:\n        if self.best_score is None:\n            self.best_score = score\n            return False\n\n        improved = (\n            score < (self.best_score - self.min_delta)\n            if self.mode == "min"\n            else score > (self.best_score + self.min_delta)\n        )\n        if improved:\n            self.best_score = score\n            self.counter = 0\n            return False\n\n        self.counter += 1\n        return self.counter >= self.patience\n'
embedded_sources['data'] = 'from __future__ import annotations\n\nimport json\nimport logging\nimport re\nfrom dataclasses import dataclass, field\nfrom itertools import product\nfrom pathlib import Path\nfrom typing import Optional\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.preprocessing import LabelEncoder\nfrom torch.utils.data import DataLoader, Dataset, Sampler\n\nfrom config import DataConfig\n\n\nLOGGER = logging.getLogger(__name__)\n\n\n@dataclass\nclass PreparedDataset:\n    features: np.ndarray\n    day_of_year: np.ndarray\n    observed_mask: np.ndarray\n    cloud_scene: Optional[np.ndarray]\n    px_count: Optional[np.ndarray]\n    labels: np.ndarray\n    parcel_ids: np.ndarray\n    parcel_tiles: np.ndarray\n    feature_names: list[str]\n    label_names: list[str]\n    time_grid: np.ndarray\n    splits: dict[str, np.ndarray]\n    group_labels: Optional[np.ndarray] = None\n    group_label_names: list[str] = field(default_factory=list)\n\n    @property\n    def num_classes(self) -> int:\n        return len(self.label_names)\n\n    @property\n    def num_features(self) -> int:\n        return len(self.feature_names)\n\n    @property\n    def seq_len(self) -> int:\n        return self.features.shape[1]\n\n    @property\n    def has_group_labels(self) -> bool:\n        return self.group_labels is not None and len(self.group_label_names) > 0\n\n    @property\n    def num_group_classes(self) -> int:\n        return len(self.group_label_names)\n\n\n@dataclass\nclass TemporalAugmentationConfig:\n    time_mask_ratio: float = 0.0\n    jitter_std: float = 0.0\n\n\nclass ParcelTimeSeriesDataset(Dataset):\n    def __init__(\n        self,\n        prepared: PreparedDataset,\n        indices: np.ndarray,\n        augmentation: Optional[TemporalAugmentationConfig] = None,\n    ) -> None:\n        self.features = prepared.features[indices]\n        self.day_of_year = prepared.day_of_year[indices]\n        self.observed_mask = prepared.observed_mask[indices]\n        default_cloud = np.full(self.day_of_year.shape, 100.0, dtype=np.float32)\n        default_px = np.zeros(self.day_of_year.shape, dtype=np.float32)\n        self.cloud_scene = (\n            prepared.cloud_scene[indices].astype(np.float32)\n            if prepared.cloud_scene is not None\n            else default_cloud\n        )\n        self.px_count = (\n            prepared.px_count[indices].astype(np.float32)\n            if prepared.px_count is not None\n            else default_px\n        )\n        self.labels = prepared.labels[indices]\n        self.group_labels = (\n            prepared.group_labels[indices]\n            if prepared.group_labels is not None\n            else np.full((len(indices),), -100, dtype=np.int64)\n        )\n        self.parcel_ids = prepared.parcel_ids[indices]\n        self.augmentation = augmentation\n\n    def __len__(self) -> int:\n        return int(self.labels.shape[0])\n\n    def __getitem__(self, idx: int) -> dict[str, torch.Tensor | str]:\n        features = self.features[idx]\n        observed_mask = self.observed_mask[idx]\n        cloud_scene = self.cloud_scene[idx]\n        px_count = self.px_count[idx]\n\n        if self.augmentation is not None:\n            features = features.copy()\n            observed_mask = observed_mask.copy()\n            cloud_scene = cloud_scene.copy()\n            px_count = px_count.copy()\n\n            observed_idx = np.flatnonzero(observed_mask)\n            if self.augmentation.time_mask_ratio > 0 and observed_idx.size > 0:\n                n_mask = int(round(observed_idx.size * self.augmentation.time_mask_ratio))\n                n_mask = max(1, min(observed_idx.size, n_mask))\n                masked_steps = np.random.choice(observed_idx, size=n_mask, replace=False)\n                features[masked_steps, :] = 0.0\n                observed_mask[masked_steps] = False\n\n            if self.augmentation.jitter_std > 0:\n                noise = np.random.normal(\n                    loc=0.0,\n                    scale=self.augmentation.jitter_std,\n                    size=features.shape,\n                ).astype(np.float32)\n                features[observed_mask] = features[observed_mask] + noise[observed_mask]\n\n        cloud_norm = np.clip(cloud_scene / 100.0, 0.0, 1.0).astype(np.float32)\n        px_norm = np.clip(np.log1p(np.clip(px_count, a_min=0.0, a_max=None)) / np.log1p(256.0), 0.0, 1.0).astype(\n            np.float32\n        )\n        cloud_norm = np.where(observed_mask, cloud_norm, 1.0).astype(np.float32)\n        px_norm = np.where(observed_mask, px_norm, 0.0).astype(np.float32)\n        reliability = (\n            observed_mask.astype(np.float32)\n            * (1.0 - cloud_norm)\n            * np.sqrt(np.clip(px_norm, a_min=0.0, a_max=1.0))\n        ).astype(np.float32)\n        quality_features = np.stack([reliability, cloud_norm, px_norm], axis=-1).astype(np.float32)\n\n        return {\n            "features": torch.from_numpy(features),\n            "day_of_year": torch.from_numpy(self.day_of_year[idx]).float(),\n            "observed_mask": torch.from_numpy(observed_mask),\n            "quality_features": torch.from_numpy(quality_features),\n            "label": torch.tensor(int(self.labels[idx]), dtype=torch.long),\n            "group_label": torch.tensor(int(self.group_labels[idx]), dtype=torch.long),\n            "parcel_id": str(self.parcel_ids[idx]),\n        }\n\n\ndef _bin_dates_to_frequency(dates: pd.Series, frequency: str) -> pd.Series:\n    """\n    Bin timestamps to a regular temporal grid.\n\n    For day-based frequencies (e.g. 5D), bins are anchored on the dataset\n    minimum date to avoid calendar-epoch artifacts.\n    """\n    freq = str(frequency).strip()\n    if not freq:\n        return dates\n\n    match = re.fullmatch(r"(?i)(\\d+)D", freq)\n    if match:\n        step_days = int(match.group(1))\n        if step_days <= 0:\n            raise ValueError(f"Invalid day frequency: {frequency}")\n        base_date = pd.to_datetime(dates.min()).normalize()\n        normalized = pd.to_datetime(dates).dt.normalize()\n        delta_days = (normalized - base_date).dt.days\n        bin_start = base_date + pd.to_timedelta((delta_days // step_days) * step_days, unit="D")\n        return pd.to_datetime(bin_start)\n\n    # Fallback for non-day frequencies.\n    try:\n        return pd.to_datetime(dates).dt.floor(freq)\n    except (ValueError, TypeError):\n        return pd.to_datetime(dates).dt.to_period(freq).dt.start_time\n\n\ndef _required_columns(cfg: DataConfig) -> list[str]:\n    return [\n        cfg.parcel_id_col,\n        cfg.date_col,\n        cfg.index_col,\n        cfg.value_col,\n        cfg.px_count_col,\n        cfg.cloud_col,\n        cfg.tile_col,\n        cfg.label_col,\n    ]\n\n\ndef _mode_or_first(series: pd.Series) -> str:\n    series = series.dropna().astype(str).str.strip()\n    series = series[series != ""]\n    if series.empty:\n        return ""\n    mode = series.mode(dropna=True)\n    if not mode.empty:\n        return str(sorted(mode.astype(str).tolist())[0])\n    return str(series.iloc[0])\n\n\ndef _validate_input_dataframe(df: pd.DataFrame, cfg: DataConfig) -> None:\n    missing = [col for col in _required_columns(cfg) if col not in df.columns]\n    if missing:\n        raise ValueError(\n            f"CSV missing required columns: {missing}. "\n            f"Expected columns include {\', \'.join(_required_columns(cfg))}."\n        )\n\n\ndef load_long_dataframe(cfg: DataConfig) -> pd.DataFrame:\n    csv_path = Path(cfg.csv_path)\n    if not csv_path.exists():\n        raise FileNotFoundError(f"CSV not found: {csv_path}")\n\n    header_cols = pd.read_csv(csv_path, nrows=0).columns.tolist()\n    missing = [col for col in _required_columns(cfg) if col not in header_cols]\n    if missing:\n        raise ValueError(\n            f"CSV missing required columns: {missing}. "\n            f"Expected columns include {\', \'.join(_required_columns(cfg))}."\n        )\n\n    usecols = list(_required_columns(cfg))\n    if cfg.label_group_col in header_cols and cfg.label_group_col not in usecols:\n        usecols.append(cfg.label_group_col)\n\n    str_dtype_cols = [cfg.parcel_id_col, cfg.index_col, cfg.label_col, cfg.tile_col]\n    if cfg.label_group_col in usecols:\n        str_dtype_cols.append(cfg.label_group_col)\n    dtype_map = {col: "string" for col in str_dtype_cols}\n\n    df = pd.read_csv(\n        csv_path,\n        usecols=usecols,\n        dtype=dtype_map,\n        low_memory=False,\n    )\n    _validate_input_dataframe(df, cfg)\n\n    df[cfg.date_col] = pd.to_datetime(df[cfg.date_col], errors="coerce")\n    if df[cfg.date_col].isna().all():\n        raise ValueError(\n            f"Column \'{cfg.date_col}\' could not be parsed as dates. "\n            "Use an ISO format such as YYYY-MM-DD."\n        )\n\n    df = df.dropna(\n        subset=[cfg.parcel_id_col, cfg.date_col, cfg.index_col, cfg.value_col, cfg.label_col]\n    ).copy()\n\n    df[cfg.parcel_id_col] = (\n        df[cfg.parcel_id_col]\n        .astype(str)\n        .str.strip()\n        .str.replace(r"\\.0+$", "", regex=True)\n    )\n    df[cfg.index_col] = df[cfg.index_col].astype(str).str.strip().str.upper()\n    df[cfg.label_col] = df[cfg.label_col].astype(str).str.strip()\n    df[cfg.tile_col] = df[cfg.tile_col].astype(str).str.strip()\n    if cfg.label_group_col in df.columns:\n        df[cfg.label_group_col] = df[cfg.label_group_col].astype(str).str.strip()\n    df[cfg.value_col] = pd.to_numeric(df[cfg.value_col], errors="coerce")\n    df[cfg.px_count_col] = pd.to_numeric(df[cfg.px_count_col], errors="coerce")\n    df[cfg.cloud_col] = pd.to_numeric(df[cfg.cloud_col], errors="coerce")\n    df = df.dropna(subset=[cfg.value_col]).copy()\n    df = df[\n        (df[cfg.parcel_id_col] != "")\n        & (df[cfg.index_col] != "")\n        & (df[cfg.label_col] != "")\n        & (df[cfg.tile_col] != "")\n    ].copy()\n\n    if cfg.index_filter:\n        allowed = {str(x).strip().upper() for x in cfg.index_filter if str(x).strip()}\n        available = set(df[cfg.index_col].dropna().astype(str).unique().tolist())\n        missing = sorted(allowed - available)\n        if missing:\n            LOGGER.warning("Requested indices not found in CSV and will be ignored: %s", ", ".join(missing))\n        before = len(df)\n        df = df[df[cfg.index_col].isin(allowed)].copy()\n        LOGGER.info("Filtered indices: %d -> %d rows", before, len(df))\n\n    if cfg.min_px_count > 0:\n        before = len(df)\n        df = df[df[cfg.px_count_col].fillna(0) >= cfg.min_px_count].copy()\n        LOGGER.info("Applied min_px_count=%d: %d -> %d rows", cfg.min_px_count, before, len(df))\n\n    if cfg.max_cloud_scene is not None:\n        before = len(df)\n        df = df[df[cfg.cloud_col].fillna(np.inf) <= cfg.max_cloud_scene].copy()\n        LOGGER.info(\n            "Applied max_cloud_scene=%.3f: %d -> %d rows",\n            cfg.max_cloud_scene,\n            before,\n            len(df),\n        )\n\n    if df.empty:\n        raise ValueError("No valid rows remaining after filtering.")\n\n    return df\n\n\ndef _aggregate_long_dataframe(df: pd.DataFrame, cfg: DataConfig) -> pd.DataFrame:\n    df_work = df.copy()\n    if cfg.time_grid_frequency:\n        n_dates_before = int(pd.to_datetime(df_work[cfg.date_col]).nunique())\n        df_work["_date_bin"] = _bin_dates_to_frequency(\n            pd.to_datetime(df_work[cfg.date_col]),\n            cfg.time_grid_frequency,\n        )\n        n_dates_after = int(df_work["_date_bin"].nunique())\n        LOGGER.info(\n            "Applied temporal binning freq=%s: unique dates %d -> %d",\n            cfg.time_grid_frequency,\n            n_dates_before,\n            n_dates_after,\n        )\n        group_date_col = "_date_bin"\n    else:\n        group_date_col = cfg.date_col\n\n    group_cols = [cfg.parcel_id_col, group_date_col, cfg.index_col]\n    agg_spec = {\n        cfg.value_col: "mean",\n        cfg.cloud_col: "mean",\n        cfg.px_count_col: "mean",\n        cfg.tile_col: _mode_or_first,\n        cfg.label_col: _mode_or_first,\n    }\n    if cfg.label_group_col in df.columns:\n        agg_spec[cfg.label_group_col] = _mode_or_first\n\n    aggregated = (\n        df_work.groupby(group_cols, dropna=False, as_index=False)\n        .agg(agg_spec)\n        .sort_values([cfg.parcel_id_col, group_date_col, cfg.index_col])\n    )\n    if group_date_col != cfg.date_col:\n        aggregated = aggregated.rename(columns={group_date_col: cfg.date_col})\n\n    if aggregated.empty:\n        raise ValueError("No rows left after grouping by parcel/date/index.")\n    return aggregated\n\n\ndef _build_time_grid(date_values: pd.Series, cfg: DataConfig) -> pd.DatetimeIndex:\n    min_date = pd.to_datetime(date_values.min())\n    max_date = pd.to_datetime(date_values.max())\n    if cfg.time_grid_frequency:\n        grid = pd.date_range(start=min_date, end=max_date, freq=cfg.time_grid_frequency)\n    else:\n        grid = pd.DatetimeIndex(sorted(pd.to_datetime(date_values).unique()))\n    if len(grid) == 0:\n        raise ValueError("Temporal grid is empty.")\n    return grid\n\n\ndef _stratify_target_if_possible(labels: np.ndarray, use_stratify: bool) -> Optional[np.ndarray]:\n    if not use_stratify:\n        return None\n    bincount = np.bincount(labels)\n    if len(bincount) < 2:\n        return None\n    if bincount.min() < 2:\n        return None\n    return labels\n\n\ndef _split_by_parcel(labels: np.ndarray, cfg: DataConfig) -> dict[str, np.ndarray]:\n    if not 0 < cfg.test_size < 1:\n        raise ValueError("test_size must be in (0, 1).")\n    if not 0 <= cfg.val_size < 1:\n        raise ValueError("val_size must be in [0, 1).")\n    if cfg.test_size + cfg.val_size >= 1:\n        raise ValueError("test_size + val_size must be < 1.")\n\n    indices = np.arange(labels.shape[0])\n    stratify_all = _stratify_target_if_possible(labels, cfg.stratify)\n\n    try:\n        train_val_idx, test_idx = train_test_split(\n            indices,\n            test_size=cfg.test_size,\n            random_state=cfg.random_state,\n            shuffle=True,\n            stratify=stratify_all,\n        )\n    except ValueError:\n        train_val_idx, test_idx = train_test_split(\n            indices,\n            test_size=cfg.test_size,\n            random_state=cfg.random_state,\n            shuffle=True,\n            stratify=None,\n        )\n\n    val_ratio_adjusted = cfg.val_size / (1.0 - cfg.test_size)\n    if val_ratio_adjusted <= 0 or len(train_val_idx) < 2:\n        return {"train": np.sort(train_val_idx), "val": np.array([], dtype=np.int64), "test": np.sort(test_idx)}\n\n    n_val = int(np.ceil(len(train_val_idx) * val_ratio_adjusted))\n    if n_val <= 0 or n_val >= len(train_val_idx):\n        return {"train": np.sort(train_val_idx), "val": np.array([], dtype=np.int64), "test": np.sort(test_idx)}\n\n    stratify_tv = _stratify_target_if_possible(labels[train_val_idx], cfg.stratify)\n    try:\n        train_idx, val_idx = train_test_split(\n            train_val_idx,\n            test_size=n_val,\n            random_state=cfg.random_state,\n            shuffle=True,\n            stratify=stratify_tv,\n        )\n    except ValueError:\n        train_idx, val_idx = train_test_split(\n            train_val_idx,\n            test_size=n_val,\n            random_state=cfg.random_state,\n            shuffle=True,\n            stratify=None,\n        )\n\n    return {"train": np.sort(train_idx), "val": np.sort(val_idx), "test": np.sort(test_idx)}\n\n\ndef _split_by_tile(labels: np.ndarray, parcel_tiles: np.ndarray, cfg: DataConfig) -> dict[str, np.ndarray]:\n    unique_tiles = np.unique(parcel_tiles)\n    if len(unique_tiles) < 3:\n        LOGGER.warning(\n            "split_method=\'tile\' requested but only %d unique tiles found. Falling back to parcel split.",\n            len(unique_tiles),\n        )\n        return _split_by_parcel(labels, cfg)\n\n    if cfg.test_size + cfg.val_size >= 1:\n        raise ValueError("test_size + val_size must be < 1.")\n\n    tile_values, tile_counts = np.unique(parcel_tiles, return_counts=True)\n    total_count = int(tile_counts.sum())\n    train_ratio = 1.0 - cfg.test_size - cfg.val_size\n    val_ratio = cfg.val_size\n    test_ratio = cfg.test_size\n    need_val = val_ratio > 0\n\n    split_ids = np.array([0, 1, 2] if need_val else [0, 2], dtype=np.int64)\n    best_score = float("inf")\n    best_assignment: Optional[np.ndarray] = None\n\n    def evaluate_assignment(assignment: np.ndarray) -> float:\n        train_count = int(tile_counts[assignment == 0].sum())\n        val_count = int(tile_counts[assignment == 1].sum()) if need_val else 0\n        test_count = int(tile_counts[assignment == 2].sum())\n\n        if train_count == 0 or test_count == 0:\n            return float("inf")\n        if need_val and val_count == 0:\n            return float("inf")\n\n        train_frac = train_count / total_count\n        val_frac = val_count / total_count if need_val else 0.0\n        test_frac = test_count / total_count\n        score = abs(train_frac - train_ratio) + abs(test_frac - test_ratio)\n        if need_val:\n            score += abs(val_frac - val_ratio)\n        return float(score)\n\n    if len(tile_values) <= 12:\n        for candidate in product(split_ids.tolist(), repeat=len(tile_values)):\n            assignment = np.asarray(candidate, dtype=np.int64)\n            score = evaluate_assignment(assignment)\n            if score < best_score:\n                best_score = score\n                best_assignment = assignment\n    else:\n        rng = np.random.default_rng(cfg.random_state)\n        for _ in range(25_000):\n            assignment = rng.choice(split_ids, size=len(tile_values), replace=True)\n            score = evaluate_assignment(assignment)\n            if score < best_score:\n                best_score = score\n                best_assignment = assignment\n\n    if best_assignment is None or not np.isfinite(best_score):\n        LOGGER.warning("Could not optimize tile split. Falling back to parcel split.")\n        return _split_by_parcel(labels, cfg)\n\n    train_tiles = tile_values[best_assignment == 0]\n    val_tiles = tile_values[best_assignment == 1] if need_val else np.array([], dtype=tile_values.dtype)\n    test_tiles = tile_values[best_assignment == 2]\n\n    train_idx = np.where(np.isin(parcel_tiles, train_tiles))[0]\n    val_idx = np.where(np.isin(parcel_tiles, val_tiles))[0]\n    test_idx = np.where(np.isin(parcel_tiles, test_tiles))[0]\n\n    if len(train_idx) == 0 or len(test_idx) == 0:\n        LOGGER.warning("Spatial split produced an empty split. Falling back to parcel split.")\n        return _split_by_parcel(labels, cfg)\n\n    LOGGER.info(\n        "Optimized tile split | train tiles=%s (%d) | val tiles=%s (%d) | test tiles=%s (%d) | score=%.4f",\n        train_tiles.tolist(),\n        len(train_idx),\n        val_tiles.tolist(),\n        len(val_idx),\n        test_tiles.tolist(),\n        len(test_idx),\n        best_score,\n    )\n\n    return {"train": np.sort(train_idx), "val": np.sort(val_idx), "test": np.sort(test_idx)}\n\n\ndef _create_splits(labels: np.ndarray, parcel_tiles: np.ndarray, cfg: DataConfig) -> dict[str, np.ndarray]:\n    if cfg.split_method == "tile":\n        return _split_by_tile(labels, parcel_tiles, cfg)\n    return _split_by_parcel(labels, cfg)\n\n\ndef _build_tensor_dataset(df: pd.DataFrame, cfg: DataConfig) -> PreparedDataset:\n    feature_names = sorted(df[cfg.index_col].astype(str).unique().tolist())\n    if not feature_names:\n        raise ValueError("No index names found in column \'%s\'." % cfg.index_col)\n\n    parcel_ids = np.array(sorted(df[cfg.parcel_id_col].astype(str).unique()))\n    time_grid = _build_time_grid(df[cfg.date_col], cfg)\n\n    pivot = df.pivot_table(\n        index=[cfg.parcel_id_col, cfg.date_col],\n        columns=cfg.index_col,\n        values=cfg.value_col,\n        aggfunc="mean",\n    )\n    pivot = pivot.reindex(columns=feature_names)\n\n    quality = (\n        df.groupby([cfg.parcel_id_col, cfg.date_col], as_index=True)\n        .agg(\n            {\n                cfg.cloud_col: "mean",\n                cfg.px_count_col: "mean",\n            }\n        )\n    )\n\n    full_index = pd.MultiIndex.from_product(\n        [parcel_ids, time_grid],\n        names=[cfg.parcel_id_col, cfg.date_col],\n    )\n    pivot = pivot.reindex(full_index)\n    quality = quality.reindex(full_index)\n\n    n_parcels = len(parcel_ids)\n    seq_len = len(time_grid)\n    num_features = len(feature_names)\n\n    observed_mask = pivot.notna().any(axis=1).to_numpy(dtype=bool).reshape(n_parcels, seq_len)\n    features = (\n        pivot.fillna(cfg.fill_value).to_numpy(dtype=np.float32).reshape(n_parcels, seq_len, num_features)\n    )\n    cloud_scene = quality[cfg.cloud_col].to_numpy(dtype=np.float32).reshape(n_parcels, seq_len)\n    px_count = quality[cfg.px_count_col].to_numpy(dtype=np.float32).reshape(n_parcels, seq_len)\n    cloud_scene = np.nan_to_num(cloud_scene, nan=100.0, posinf=100.0, neginf=100.0)\n    px_count = np.nan_to_num(px_count, nan=0.0, posinf=0.0, neginf=0.0)\n    cloud_scene = np.clip(cloud_scene, a_min=0.0, a_max=100.0).astype(np.float32)\n    px_count = np.clip(px_count, a_min=0.0, a_max=None).astype(np.float32)\n\n    day_vector = np.asarray(time_grid.dayofyear, dtype=np.int16)\n    day_of_year = np.tile(day_vector[None, :], (n_parcels, 1))\n\n    metadata_spec = {cfg.label_col: _mode_or_first, cfg.tile_col: _mode_or_first}\n    if cfg.label_group_col in df.columns:\n        metadata_spec[cfg.label_group_col] = _mode_or_first\n\n    parcel_metadata = (\n        df.groupby(cfg.parcel_id_col, as_index=True)\n        .agg(metadata_spec)\n        .reindex(parcel_ids)\n    )\n    if parcel_metadata[cfg.label_col].isna().any():\n        raise ValueError("Missing labels for some parcels after aggregation.")\n\n    label_encoder = LabelEncoder()\n    labels = label_encoder.fit_transform(parcel_metadata[cfg.label_col].astype(str).to_numpy())\n    label_names = label_encoder.classes_.tolist()\n    parcel_tiles = parcel_metadata[cfg.tile_col].fillna("unknown").astype(str).to_numpy()\n\n    group_labels: Optional[np.ndarray] = None\n    group_label_names: list[str] = []\n    if cfg.label_group_col in parcel_metadata.columns:\n        group_series = parcel_metadata[cfg.label_group_col].fillna("").astype(str).str.strip()\n        valid_group_mask = group_series != ""\n        if valid_group_mask.any():\n            group_encoder = LabelEncoder()\n            group_encoder.fit(group_series[valid_group_mask].to_numpy())\n            group_labels = np.full((len(parcel_ids),), -100, dtype=np.int64)\n            group_labels[valid_group_mask.to_numpy()] = group_encoder.transform(\n                group_series[valid_group_mask].to_numpy()\n            )\n            group_label_names = group_encoder.classes_.tolist()\n            LOGGER.info(\n                "Detected %d/%d parcels with group labels in \'%s\' (%d groups).",\n                int(valid_group_mask.sum()),\n                int(len(valid_group_mask)),\n                cfg.label_group_col,\n                len(group_label_names),\n            )\n        else:\n            LOGGER.info(\n                "Column \'%s\' exists but has no non-empty values. Group task will be disabled.",\n                cfg.label_group_col,\n            )\n\n    min_obs_mask = observed_mask.sum(axis=1) >= cfg.min_obs_per_parcel\n    if min_obs_mask.sum() == 0:\n        raise ValueError(\n            "All parcels were removed by min_obs_per_parcel=%d. Lower this threshold."\n            % cfg.min_obs_per_parcel\n        )\n    if min_obs_mask.sum() < len(min_obs_mask):\n        LOGGER.info(\n            "Removed %d parcels with fewer than %d observed dates.",\n            int((~min_obs_mask).sum()),\n            cfg.min_obs_per_parcel,\n        )\n\n    features = features[min_obs_mask]\n    day_of_year = day_of_year[min_obs_mask]\n    observed_mask = observed_mask[min_obs_mask]\n    cloud_scene = cloud_scene[min_obs_mask]\n    px_count = px_count[min_obs_mask]\n    labels = labels[min_obs_mask]\n    parcel_ids = parcel_ids[min_obs_mask]\n    parcel_tiles = parcel_tiles[min_obs_mask]\n    if group_labels is not None:\n        group_labels = group_labels[min_obs_mask]\n\n    splits = _create_splits(labels=labels, parcel_tiles=parcel_tiles, cfg=cfg)\n\n    return PreparedDataset(\n        features=features.astype(np.float32),\n        day_of_year=day_of_year.astype(np.int16),\n        observed_mask=observed_mask.astype(bool),\n        cloud_scene=cloud_scene.astype(np.float32),\n        px_count=px_count.astype(np.float32),\n        labels=labels.astype(np.int64),\n        parcel_ids=parcel_ids.astype(str),\n        parcel_tiles=parcel_tiles.astype(str),\n        feature_names=feature_names,\n        label_names=label_names,\n        group_labels=group_labels.astype(np.int64) if group_labels is not None else None,\n        group_label_names=group_label_names,\n        time_grid=np.asarray(time_grid.values, dtype="datetime64[ns]"),\n        splits=splits,\n    )\n\n\ndef prepare_dataset(cfg: DataConfig) -> PreparedDataset:\n    if cfg.prepared_npz_path:\n        npz_path = Path(cfg.prepared_npz_path)\n        if not npz_path.exists():\n            raise FileNotFoundError(f"Prepared dataset not found: {npz_path}")\n        prepared = load_prepared_dataset(npz_path)\n    else:\n        df = load_long_dataframe(cfg)\n        aggregated = _aggregate_long_dataframe(df, cfg)\n        prepared = _build_tensor_dataset(aggregated, cfg)\n\n    if cfg.save_prepared_npz_path:\n        save_prepared_dataset(prepared, cfg.save_prepared_npz_path)\n\n    return prepared\n\n\ndef standardize_prepared_features(\n    prepared: PreparedDataset,\n    train_indices: np.ndarray,\n    eps: float = 1e-6,\n) -> dict[str, np.ndarray]:\n    """\n    Standardize features using only train split statistics.\n    Missing dates are excluded from stats via observed_mask and reset to 0 after scaling.\n    """\n    if train_indices.size == 0:\n        raise ValueError("Cannot standardize features with an empty train split.")\n\n    train_features = prepared.features[train_indices]  # [N_train, T, F]\n    train_observed = prepared.observed_mask[train_indices][..., None].astype(np.float32)  # [N_train, T, 1]\n\n    counts = train_observed.sum(axis=(0, 1)).reshape(1).repeat(prepared.num_features)\n    if np.any(counts <= 0):\n        raise ValueError("Found a feature with zero observed samples in train split.")\n\n    mean = (train_features * train_observed).sum(axis=(0, 1)) / np.clip(counts, a_min=1.0, a_max=None)\n    var = (\n        ((train_features - mean[None, None, :]) ** 2) * train_observed\n    ).sum(axis=(0, 1)) / np.clip(counts, a_min=1.0, a_max=None)\n    std = np.sqrt(np.clip(var, a_min=eps**2, a_max=None))\n\n    scaled = (prepared.features - mean[None, None, :]) / std[None, None, :]\n    scaled[~prepared.observed_mask] = 0.0\n    prepared.features = scaled.astype(np.float32)\n\n    return {\n        "mean": mean.astype(np.float32),\n        "std": std.astype(np.float32),\n        "count_per_feature": counts.astype(np.int64),\n    }\n\n\ndef build_dataloaders(\n    prepared: PreparedDataset,\n    batch_size: int,\n    num_workers: int = 0,\n    pin_memory: bool = False,\n    train_sampler: Optional[Sampler] = None,\n    train_augmentation: Optional[TemporalAugmentationConfig] = None,\n) -> dict[str, DataLoader]:\n    dataloaders: dict[str, DataLoader] = {}\n    for split_name, split_indices in prepared.splits.items():\n        split_dataset = ParcelTimeSeriesDataset(\n            prepared,\n            split_indices,\n            augmentation=train_augmentation if split_name == "train" else None,\n        )\n        use_sampler = split_name == "train" and train_sampler is not None\n        dataloaders[split_name] = DataLoader(\n            split_dataset,\n            batch_size=batch_size,\n            shuffle=(split_name == "train" and not use_sampler),\n            sampler=train_sampler if use_sampler else None,\n            num_workers=num_workers,\n            pin_memory=pin_memory,\n            drop_last=False,\n        )\n    return dataloaders\n\n\ndef save_prepared_dataset(prepared: PreparedDataset, path: str | Path) -> None:\n    save_path = Path(path)\n    save_path.parent.mkdir(parents=True, exist_ok=True)\n\n    splits_json = json.dumps({k: v.tolist() for k, v in prepared.splits.items()})\n    has_group_labels = prepared.group_labels is not None and len(prepared.group_label_names) > 0\n    cloud_scene = (\n        prepared.cloud_scene.astype(np.float32)\n        if prepared.cloud_scene is not None\n        else np.full(prepared.observed_mask.shape, 100.0, dtype=np.float32)\n    )\n    px_count = (\n        prepared.px_count.astype(np.float32)\n        if prepared.px_count is not None\n        else np.zeros(prepared.observed_mask.shape, dtype=np.float32)\n    )\n    np.savez_compressed(\n        save_path,\n        features=prepared.features,\n        day_of_year=prepared.day_of_year,\n        observed_mask=prepared.observed_mask,\n        cloud_scene=cloud_scene,\n        px_count=px_count,\n        labels=prepared.labels,\n        group_labels=prepared.group_labels if has_group_labels else np.array([], dtype=np.int64),\n        parcel_ids=prepared.parcel_ids,\n        parcel_tiles=prepared.parcel_tiles,\n        feature_names=np.asarray(prepared.feature_names, dtype=object),\n        label_names=np.asarray(prepared.label_names, dtype=object),\n        group_label_names=np.asarray(prepared.group_label_names, dtype=object),\n        time_grid=prepared.time_grid.astype("datetime64[ns]").astype("int64"),\n        splits_json=np.asarray([splits_json], dtype=object),\n        has_group_labels=np.asarray([has_group_labels], dtype=np.bool_),\n    )\n    LOGGER.info("Saved prepared dataset to %s", save_path)\n\n\ndef load_prepared_dataset(path: str | Path) -> PreparedDataset:\n    load_path = Path(path)\n    if not load_path.exists():\n        raise FileNotFoundError(f"NPZ dataset not found: {load_path}")\n\n    with np.load(load_path, allow_pickle=True) as data:\n        splits_json = str(data["splits_json"][0])\n        splits = {k: np.asarray(v, dtype=np.int64) for k, v in json.loads(splits_json).items()}\n        has_group_labels = bool(data["has_group_labels"][0]) if "has_group_labels" in data.files else False\n        if has_group_labels and "group_labels" in data.files and "group_label_names" in data.files:\n            group_labels = data["group_labels"].astype(np.int64)\n            group_label_names = [str(x) for x in data["group_label_names"].tolist()]\n            if group_labels.shape[0] == data["labels"].shape[0] and len(group_label_names) > 0:\n                parsed_group_labels: Optional[np.ndarray] = group_labels\n                parsed_group_label_names = group_label_names\n            else:\n                parsed_group_labels = None\n                parsed_group_label_names = []\n        else:\n            parsed_group_labels = None\n            parsed_group_label_names = []\n\n        prepared = PreparedDataset(\n            features=data["features"].astype(np.float32),\n            day_of_year=data["day_of_year"].astype(np.int16),\n            observed_mask=data["observed_mask"].astype(bool),\n            cloud_scene=(\n                data["cloud_scene"].astype(np.float32)\n                if "cloud_scene" in data.files\n                else np.full(data["observed_mask"].shape, 100.0, dtype=np.float32)\n            ),\n            px_count=(\n                data["px_count"].astype(np.float32)\n                if "px_count" in data.files\n                else np.zeros(data["observed_mask"].shape, dtype=np.float32)\n            ),\n            labels=data["labels"].astype(np.int64),\n            parcel_ids=data["parcel_ids"].astype(str),\n            parcel_tiles=data["parcel_tiles"].astype(str),\n            feature_names=[str(x) for x in data["feature_names"].tolist()],\n            label_names=[str(x) for x in data["label_names"].tolist()],\n            group_labels=parsed_group_labels,\n            group_label_names=parsed_group_label_names,\n            time_grid=data["time_grid"].astype("int64").astype("datetime64[ns]"),\n            splits=splits,\n        )\n    return prepared\n'
embedded_sources['model'] = 'from __future__ import annotations\n\nimport math\nfrom typing import Optional\n\nimport torch\nfrom torch import Tensor, nn\n\nfrom config import ModelConfig\n\n\nclass DayOfYearEncoding(nn.Module):\n    """Fourier day-of-year encoding projected into model space."""\n\n    def __init__(self, d_model: int, dropout: float) -> None:\n        super().__init__()\n        self.proj = nn.Sequential(\n            nn.Linear(2, d_model),\n            nn.GELU(),\n            nn.Dropout(dropout),\n        )\n\n    def forward(self, day_of_year: Tensor) -> Tensor:\n        # day_of_year: [B, T] in [1, 366]\n        doy = day_of_year.clamp(min=1.0, max=366.0)\n        angle = 2.0 * math.pi * (doy / 366.0)\n        sinusoidal = torch.stack([torch.sin(angle), torch.cos(angle)], dim=-1)\n        return self.proj(sinusoidal)\n\n\nclass TemporalEncoderLayer(nn.Module):\n    def __init__(\n        self,\n        d_model: int,\n        n_heads: int,\n        dim_feedforward: int,\n        dropout: float,\n        norm_first: bool,\n    ) -> None:\n        super().__init__()\n        self.self_attn = nn.MultiheadAttention(\n            embed_dim=d_model,\n            num_heads=n_heads,\n            dropout=dropout,\n            batch_first=True,\n        )\n        self.linear1 = nn.Linear(d_model, dim_feedforward)\n        self.linear2 = nn.Linear(dim_feedforward, d_model)\n        self.dropout = nn.Dropout(dropout)\n        self.dropout1 = nn.Dropout(dropout)\n        self.dropout2 = nn.Dropout(dropout)\n        self.norm1 = nn.LayerNorm(d_model)\n        self.norm2 = nn.LayerNorm(d_model)\n        self.activation = nn.GELU()\n        self.norm_first = norm_first\n\n    def _self_attn_block(\n        self,\n        x: Tensor,\n        key_padding_mask: Optional[Tensor],\n        return_attention: bool,\n    ) -> tuple[Tensor, Optional[Tensor]]:\n        attn_output, attn_weights = self.self_attn(\n            x,\n            x,\n            x,\n            key_padding_mask=key_padding_mask,\n            need_weights=return_attention,\n            average_attn_weights=False,\n        )\n        return self.dropout1(attn_output), attn_weights if return_attention else None\n\n    def _ff_block(self, x: Tensor) -> Tensor:\n        x = self.linear2(self.dropout(self.activation(self.linear1(x))))\n        return self.dropout2(x)\n\n    def forward(\n        self,\n        x: Tensor,\n        key_padding_mask: Optional[Tensor] = None,\n        return_attention: bool = False,\n    ) -> tuple[Tensor, Optional[Tensor]]:\n        if self.norm_first:\n            attn_out, attn_weights = self._self_attn_block(\n                self.norm1(x), key_padding_mask=key_padding_mask, return_attention=return_attention\n            )\n            x = x + attn_out\n            x = x + self._ff_block(self.norm2(x))\n            return x, attn_weights\n\n        attn_out, attn_weights = self._self_attn_block(\n            x, key_padding_mask=key_padding_mask, return_attention=return_attention\n        )\n        x = self.norm1(x + attn_out)\n        x = self.norm2(x + self._ff_block(x))\n        return x, attn_weights\n\n\nclass TemporalTransformerClassifier(nn.Module):\n    def __init__(\n        self,\n        input_dim: int,\n        num_classes: int,\n        cfg: ModelConfig,\n        num_group_classes: int = 0,\n    ) -> None:\n        super().__init__()\n        if cfg.pooling not in {"cls", "mean"}:\n            raise ValueError(f"Unsupported pooling mode: {cfg.pooling}")\n\n        self.input_dim = input_dim\n        self.num_classes = num_classes\n        self.num_group_classes = num_group_classes\n        self.pooling = cfg.pooling\n        self.reliability_aware = bool(getattr(cfg, "reliability_aware", False))\n        self.feature_proj = nn.Linear(input_dim, cfg.d_model)\n        self.doy_encoding = DayOfYearEncoding(cfg.d_model, cfg.dropout)\n        self.input_dropout = nn.Dropout(cfg.dropout)\n        if self.reliability_aware:\n            self.reliability_proj = nn.Sequential(\n                nn.Linear(3, cfg.d_model),\n                nn.GELU(),\n                nn.Linear(cfg.d_model, cfg.d_model),\n            )\n            self.reliability_dropout = nn.Dropout(cfg.dropout)\n            self.reliability_gate_logit = nn.Parameter(torch.tensor(0.0, dtype=torch.float32))\n        else:\n            self.reliability_proj = None\n            self.reliability_dropout = None\n            self.register_parameter("reliability_gate_logit", None)\n        self.layers = nn.ModuleList(\n            [\n                TemporalEncoderLayer(\n                    d_model=cfg.d_model,\n                    n_heads=cfg.n_heads,\n                    dim_feedforward=cfg.dim_feedforward,\n                    dropout=cfg.dropout,\n                    norm_first=cfg.use_layer_norm_first,\n                )\n                for _ in range(cfg.n_layers)\n            ]\n        )\n\n        if self.pooling == "cls":\n            self.cls_token = nn.Parameter(torch.zeros(1, 1, cfg.d_model))\n            nn.init.normal_(self.cls_token, mean=0.0, std=0.02)\n        else:\n            self.register_parameter("cls_token", None)\n\n        self.classifier = nn.Sequential(\n            nn.LayerNorm(cfg.d_model),\n            nn.Linear(cfg.d_model, num_classes),\n        )\n        self.group_classifier = (\n            nn.Sequential(\n                nn.LayerNorm(cfg.d_model),\n                nn.Linear(cfg.d_model, num_group_classes),\n            )\n            if num_group_classes > 0\n            else None\n        )\n        if num_group_classes > 0:\n            compat_default = torch.ones((num_group_classes, num_classes), dtype=torch.float32)\n        else:\n            compat_default = torch.empty((0, 0), dtype=torch.float32)\n\n        # Non-persistent buffers: moved with .to(device), but not saved in checkpoints.\n        # This keeps backward compatibility with older checkpoints.\n        self.register_buffer("class_group_compat", compat_default, persistent=False)\n        self.register_buffer("hierarchical_constraint_enabled", torch.tensor(False), persistent=False)\n        self.register_buffer("hierarchical_constraint_weight", torch.tensor(0.0, dtype=torch.float32), persistent=False)\n        self.register_buffer("hierarchical_constraint_eps", torch.tensor(1.0e-6, dtype=torch.float32), persistent=False)\n\n    def _prepare_quality_inputs(\n        self,\n        observed_mask: Tensor,\n        quality_features: Optional[Tensor],\n        dtype: torch.dtype,\n    ) -> tuple[Tensor, Tensor]:\n        observed = observed_mask.float()\n        if quality_features is None:\n            cloud = 1.0 - observed\n            px = observed\n            quality = torch.stack([observed, cloud, px], dim=-1)\n        else:\n            quality = quality_features.to(device=observed_mask.device, dtype=dtype)\n            if quality.ndim != 3:\n                raise ValueError("quality_features must have shape [B, T, Q].")\n            if quality.shape[0] != observed_mask.shape[0] or quality.shape[1] != observed_mask.shape[1]:\n                raise ValueError(\n                    "quality_features must match features/observed_mask shape on the first two dimensions."\n                )\n            if quality.shape[-1] < 3:\n                pad = torch.zeros(\n                    quality.shape[0],\n                    quality.shape[1],\n                    3 - quality.shape[-1],\n                    device=quality.device,\n                    dtype=quality.dtype,\n                )\n                quality = torch.cat([quality, pad], dim=-1)\n            elif quality.shape[-1] > 3:\n                quality = quality[..., :3]\n\n        quality = quality.clone()\n        quality[..., 1] = torch.where(observed_mask, quality[..., 1].clamp(0.0, 1.0), torch.ones_like(quality[..., 1]))\n        quality[..., 2] = torch.where(observed_mask, quality[..., 2].clamp(0.0, 1.0), torch.zeros_like(quality[..., 2]))\n        reliability = torch.where(observed_mask, quality[..., 0].clamp(0.0, 1.0), torch.zeros_like(quality[..., 0]))\n        quality[..., 0] = reliability\n        return quality, reliability\n\n    def encode(\n        self,\n        features: Tensor,\n        day_of_year: Tensor,\n        observed_mask: Tensor,\n        quality_features: Optional[Tensor] = None,\n        return_attention: bool = False,\n    ) -> dict[str, Tensor | list[Tensor]]:\n        key_padding_mask = ~observed_mask.bool()  # True means "ignore token"\n\n        x = self.feature_proj(features) + self.doy_encoding(day_of_year)\n        temporal_reliability = observed_mask.float()\n        if self.reliability_aware:\n            quality_inputs, temporal_reliability = self._prepare_quality_inputs(\n                observed_mask=observed_mask.bool(),\n                quality_features=quality_features,\n                dtype=features.dtype,\n            )\n            reliability_embed = self.reliability_proj(quality_inputs)\n            if self.reliability_dropout is not None:\n                reliability_embed = self.reliability_dropout(reliability_embed)\n            gate_strength = torch.sigmoid(self.reliability_gate_logit)\n            x = x + reliability_embed\n            token_gate = (1.0 - gate_strength) + (gate_strength * temporal_reliability.unsqueeze(-1))\n            x = x * token_gate\n        x = self.input_dropout(x)\n\n        if self.pooling == "cls":\n            cls = self.cls_token.expand(features.shape[0], -1, -1)\n            x = torch.cat([cls, x], dim=1)\n            cls_pad = torch.zeros((features.shape[0], 1), dtype=torch.bool, device=features.device)\n            key_padding_mask = torch.cat([cls_pad, key_padding_mask], dim=1)\n\n        attentions: list[Tensor] = []\n        for layer in self.layers:\n            x, attn_weights = layer(\n                x,\n                key_padding_mask=key_padding_mask,\n                return_attention=return_attention,\n            )\n            if return_attention and attn_weights is not None:\n                attentions.append(attn_weights)\n\n        temporal_tokens = x[:, 1:] if self.pooling == "cls" else x\n        return {\n            "encoded_tokens": x,\n            "temporal_tokens": temporal_tokens,\n            "key_padding_mask": key_padding_mask,\n            "temporal_reliability": temporal_reliability,\n            "attention_maps": attentions,\n        }\n\n    def configure_hierarchical_constraint(\n        self,\n        class_group_compat: Tensor,\n        weight: float,\n        eps: float = 1.0e-6,\n        enabled: bool = True,\n    ) -> None:\n        if self.num_group_classes <= 0:\n            raise ValueError("Cannot enable hierarchical constraint without group head.")\n        if class_group_compat.ndim != 2:\n            raise ValueError("class_group_compat must be a 2D tensor [G, C].")\n        if class_group_compat.shape[0] != self.num_group_classes:\n            raise ValueError(\n                f"class_group_compat has invalid group dimension: {class_group_compat.shape[0]} "\n                f"(expected {self.num_group_classes})."\n            )\n        if class_group_compat.shape[1] != self.num_classes:\n            raise ValueError(\n                f"class_group_compat has invalid class dimension: {class_group_compat.shape[1]} "\n                f"(expected {self.num_classes})."\n            )\n        if eps <= 0:\n            raise ValueError("eps must be > 0.")\n\n        compat = class_group_compat.to(device=self.class_group_compat.device, dtype=torch.float32)\n        compat = compat.clamp(min=0.0)\n        col_sum = compat.sum(dim=0, keepdim=True)\n        zero_cols = col_sum <= 0.0\n        if torch.any(zero_cols):\n            compat[:, zero_cols.squeeze(0)] = 1.0 / float(self.num_group_classes)\n            col_sum = compat.sum(dim=0, keepdim=True)\n        compat = compat / col_sum.clamp(min=1.0e-12)\n\n        self.class_group_compat.copy_(compat)\n        self.hierarchical_constraint_weight.fill_(float(weight))\n        self.hierarchical_constraint_eps.fill_(float(eps))\n        self.hierarchical_constraint_enabled.fill_(bool(enabled))\n\n    def forward(\n        self,\n        features: Tensor,\n        day_of_year: Tensor,\n        observed_mask: Tensor,\n        quality_features: Optional[Tensor] = None,\n        return_attention: bool = False,\n    ) -> dict[str, Tensor | list[Tensor]]:\n        """\n        Args:\n            features: [B, T, F]\n            day_of_year: [B, T]\n            observed_mask: [B, T] with True where date exists.\n        """\n        encoded = self.encode(\n            features=features,\n            day_of_year=day_of_year,\n            observed_mask=observed_mask,\n            quality_features=quality_features,\n            return_attention=return_attention,\n        )\n        x = encoded["encoded_tokens"]\n        key_padding_mask = encoded["key_padding_mask"]\n        temporal_reliability = encoded["temporal_reliability"]\n        attentions = encoded["attention_maps"]\n\n        if self.pooling == "cls":\n            pooled = x[:, 0]\n        else:\n            valid_tokens = (~key_padding_mask).unsqueeze(-1).float()\n            if self.reliability_aware:\n                rel_weights = temporal_reliability.unsqueeze(-1).clamp(min=0.05)\n                token_weights = valid_tokens * rel_weights\n            else:\n                token_weights = valid_tokens\n            denominator = token_weights.sum(dim=1).clamp(min=1.0)\n            pooled = (x * token_weights).sum(dim=1) / denominator\n\n        logits = self.classifier(pooled)\n        outputs: dict[str, Tensor | list[Tensor]] = {"logits": logits}\n        if self.group_classifier is not None:\n            group_logits = self.group_classifier(pooled)\n            outputs["group_logits"] = group_logits\n\n            if bool(self.hierarchical_constraint_enabled.item()):\n                group_probs = torch.softmax(group_logits, dim=1)\n                class_compat = torch.matmul(group_probs, self.class_group_compat)\n                constrained_logits = logits + float(self.hierarchical_constraint_weight.item()) * torch.log(\n                    class_compat.clamp(min=float(self.hierarchical_constraint_eps.item()))\n                )\n                outputs["logits_raw"] = logits\n                outputs["logits"] = constrained_logits\n                outputs["hierarchical_class_compat"] = class_compat\n        if self.reliability_aware:\n            outputs["temporal_reliability"] = temporal_reliability\n        if return_attention:\n            outputs["attention_maps"] = attentions\n        return outputs\n'
embedded_sources['evaluate'] = 'from __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\nfrom typing import Any, Optional\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.metrics import (\n    accuracy_score,\n    classification_report,\n    confusion_matrix,\n    f1_score,\n    precision_score,\n    recall_score,\n)\nfrom torch.utils.data import DataLoader\n\nfrom config import get_default_config\nfrom data import build_dataloaders, prepare_dataset, standardize_prepared_features\nfrom model import TemporalTransformerClassifier\nfrom utils import resolve_device, save_json, setup_logger\n\n\ndef configure_hierarchical_constraint_from_metadata(\n    model: TemporalTransformerClassifier,\n    metadata: dict[str, Any],\n    logger,\n) -> None:\n    enabled = bool(metadata.get("hierarchical_constraint", False))\n    compat = metadata.get("class_group_compat")\n    if not enabled or compat is None:\n        return\n\n    try:\n        compat_tensor = torch.tensor(compat, dtype=torch.float32, device=next(model.parameters()).device)\n        model.configure_hierarchical_constraint(\n            class_group_compat=compat_tensor,\n            weight=float(metadata.get("hierarchical_constraint_weight", 1.0)),\n            eps=float(metadata.get("hierarchical_constraint_eps", 1.0e-6)),\n            enabled=True,\n        )\n        logger.info(\n            "Enabled hierarchical constrained decoding from checkpoint metadata (weight=%.3f).",\n            float(metadata.get("hierarchical_constraint_weight", 1.0)),\n        )\n    except Exception as exc:\n        logger.warning("Could not apply hierarchical constraint from metadata: %s", exc)\n\n\ndef checkpoint_uses_reliability(checkpoint: dict[str, Any]) -> bool:\n    metadata = checkpoint.get("metadata", {})\n    if isinstance(metadata, dict) and "reliability_aware" in metadata:\n        return bool(metadata.get("reliability_aware", False))\n\n    state_dict = checkpoint.get("model_state_dict", {})\n    if not isinstance(state_dict, dict):\n        return False\n\n    for raw_key in state_dict.keys():\n        key = str(raw_key)\n        if key.startswith("module."):\n            key = key[len("module.") :]\n        if key.startswith("backbone."):\n            key = key[len("backbone.") :]\n        if key.startswith("reliability_proj.") or key == "reliability_gate_logit":\n            return True\n    return False\n\n\n@torch.no_grad()\ndef predict_model(\n    model: torch.nn.Module,\n    dataloader: DataLoader,\n    device: torch.device,\n    pooling: str,\n    return_attention: bool = False,\n) -> dict[str, Any]:\n    model.eval()\n    all_logits: list[np.ndarray] = []\n    all_labels: list[np.ndarray] = []\n    parcel_ids: list[str] = []\n\n    attention_sum: Optional[np.ndarray] = None\n    attention_count: Optional[np.ndarray] = None\n\n    for batch in dataloader:\n        features = batch["features"].to(device=device, dtype=torch.float32)\n        day_of_year = batch["day_of_year"].to(device=device, dtype=torch.float32)\n        observed_mask = batch["observed_mask"].to(device=device, dtype=torch.bool)\n        quality_features = batch["quality_features"].to(device=device, dtype=torch.float32)\n        labels = batch["label"].to(device=device, dtype=torch.long)\n\n        outputs = model(\n            features=features,\n            day_of_year=day_of_year,\n            observed_mask=observed_mask,\n            quality_features=quality_features,\n            return_attention=return_attention,\n        )\n        logits = outputs["logits"]\n\n        all_logits.append(logits.cpu().numpy())\n        all_labels.append(labels.cpu().numpy())\n        parcel_ids.extend(batch["parcel_id"])\n\n        if return_attention and outputs.get("attention_maps"):\n            last_layer = outputs["attention_maps"][-1]  # [B, H, S, S]\n            attention = _reduce_attention_to_dates(\n                attention_maps=last_layer,\n                observed_mask=observed_mask,\n                pooling=pooling,\n            )\n            batch_sum = attention.sum(axis=0)\n            batch_count = observed_mask.sum(dim=0).cpu().numpy().astype(np.float64)\n            if attention_sum is None:\n                attention_sum = batch_sum\n                attention_count = batch_count\n            else:\n                attention_sum += batch_sum\n                attention_count += batch_count\n\n    logits_np = np.concatenate(all_logits, axis=0) if all_logits else np.empty((0, 0))\n    labels_np = np.concatenate(all_labels, axis=0) if all_labels else np.empty((0,), dtype=np.int64)\n    preds_np = logits_np.argmax(axis=1) if logits_np.size else np.empty((0,), dtype=np.int64)\n\n    output: dict[str, Any] = {\n        "logits": logits_np,\n        "labels": labels_np,\n        "preds": preds_np,\n        "parcel_ids": np.asarray(parcel_ids, dtype=object),\n    }\n    if return_attention and attention_sum is not None and attention_count is not None:\n        temporal_importance = attention_sum / np.clip(attention_count, a_min=1e-6, a_max=None)\n        output["temporal_attention"] = temporal_importance\n    return output\n\n\ndef _reduce_attention_to_dates(\n    attention_maps: torch.Tensor,\n    observed_mask: torch.Tensor,\n    pooling: str,\n) -> np.ndarray:\n    """\n    Convert last-layer attention maps to per-date importance scores.\n    """\n    attn = attention_maps.mean(dim=1)  # [B, S, S], average over heads\n    mask = observed_mask.float()\n\n    if pooling == "cls":\n        # CLS token attends to temporal tokens.\n        cls_to_dates = attn[:, 0, 1:]\n        score = cls_to_dates * mask\n        return score.cpu().numpy()\n\n    # Mean pooling: average attention received by each date token.\n    token_to_token = attn\n    score = token_to_token.mean(dim=1)\n    score = score * mask\n    return score.cpu().numpy()\n\n\ndef compute_classification_metrics(\n    y_true: np.ndarray,\n    y_pred: np.ndarray,\n    label_names: list[str],\n) -> dict[str, Any]:\n    metrics = {\n        "accuracy": float(accuracy_score(y_true, y_pred)),\n        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),\n        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),\n        "recall_weighted": float(recall_score(y_true, y_pred, average="weighted", zero_division=0)),\n        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),\n        "f1_weighted": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),\n    }\n    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(label_names)))\n    report_dict = classification_report(\n        y_true,\n        y_pred,\n        labels=np.arange(len(label_names)),\n        target_names=label_names,\n        zero_division=0,\n        output_dict=True,\n    )\n    report_text = classification_report(\n        y_true,\n        y_pred,\n        labels=np.arange(len(label_names)),\n        target_names=label_names,\n        zero_division=0,\n    )\n\n    metrics["confusion_matrix"] = cm\n    metrics["classification_report_dict"] = report_dict\n    metrics["classification_report_text"] = report_text\n    return metrics\n\n\ndef analyze_errors_by_class(\n    confusion: np.ndarray,\n    label_names: list[str],\n) -> list[dict[str, Any]]:\n    analysis: list[dict[str, Any]] = []\n    num_classes = confusion.shape[0]\n    for class_idx in range(num_classes):\n        support = int(confusion[class_idx, :].sum())\n        tp = int(confusion[class_idx, class_idx])\n        fn = int(support - tp)\n\n        off_diag = confusion[class_idx, :].copy()\n        off_diag[class_idx] = 0\n        if off_diag.sum() > 0:\n            most_confused_idx = int(np.argmax(off_diag))\n            most_confused_label = label_names[most_confused_idx]\n            most_confused_count = int(off_diag[most_confused_idx])\n        else:\n            most_confused_label = None\n            most_confused_count = 0\n\n        analysis.append(\n            {\n                "class_name": label_names[class_idx],\n                "support": support,\n                "true_positive": tp,\n                "false_negative": fn,\n                "main_confusion_target": most_confused_label,\n                "main_confusion_count": most_confused_count,\n            }\n        )\n    return analysis\n\n\ndef plot_confusion_matrix(\n    confusion: np.ndarray,\n    label_names: list[str],\n    output_path: str | Path,\n    normalize: bool = False,\n) -> None:\n    cm = confusion.astype(np.float64)\n    if normalize:\n        row_sum = cm.sum(axis=1, keepdims=True)\n        df_norm = cm / np.clip(row_sum, a_min=1e-12, a_max=None)\n    else:\n        df_norm = cm\n\n    plt.figure(figsize=(8, 6))\n    plt.imshow(df_norm, cmap="Greens")\n    plt.colorbar()\n    plt.xticks(np.arange(len(label_names)), label_names, rotation=90)\n    plt.yticks(np.arange(len(label_names)), label_names)\n    plt.title("Matrice de confusion" + (" (normalisée)" if normalize else ""))\n    plt.xlabel("Classe prédite")\n    plt.ylabel("Classe réelle")\n    ax = plt.gca()\n    ax.set_frame_on(False)\n    for spine in ax.spines.values():\n        spine.set_visible(False)\n    ax.tick_params(length=0)\n    plt.tight_layout()\n    output_path = Path(output_path)\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    plt.savefig(output_path, dpi=200)\n    plt.close()\n\n\ndef evaluate_split(\n    model: torch.nn.Module,\n    dataloader: DataLoader,\n    device: torch.device,\n    label_names: list[str],\n    pooling: str,\n    return_attention: bool = False,\n) -> dict[str, Any]:\n    if len(dataloader.dataset) == 0:\n        raise ValueError("Impossible d\'évaluer un split vide.")\n\n    predictions = predict_model(\n        model=model,\n        dataloader=dataloader,\n        device=device,\n        pooling=pooling,\n        return_attention=return_attention,\n    )\n    metrics = compute_classification_metrics(\n        y_true=predictions["labels"],\n        y_pred=predictions["preds"],\n        label_names=label_names,\n    )\n    metrics["predictions"] = predictions\n    metrics["error_analysis"] = analyze_errors_by_class(\n        confusion=metrics["confusion_matrix"],\n        label_names=label_names,\n    )\n    return metrics\n\n\ndef _serialize_metrics(metrics: dict[str, Any]) -> dict[str, Any]:\n    return {\n        "exactitude": metrics["accuracy"],\n        "precision_macro": metrics["precision_macro"],\n        "rappel_macro": metrics["recall_macro"],\n        "rappel_pondere": metrics["recall_weighted"],\n        "f1_macro": metrics["f1_macro"],\n        "f1_pondere": metrics["f1_weighted"],\n        "rapport_classification": metrics["classification_report_dict"],\n        "analyse_erreurs": metrics["error_analysis"],\n        "matrice_confusion": metrics["confusion_matrix"].tolist(),\n        # Compatibilite descendante\n        "accuracy": metrics["accuracy"],\n        "recall_macro": metrics["recall_macro"],\n        "recall_weighted": metrics["recall_weighted"],\n        "f1_weighted": metrics["f1_weighted"],\n        "classification_report": metrics["classification_report_dict"],\n        "error_analysis": metrics["error_analysis"],\n        "confusion_matrix": metrics["confusion_matrix"].tolist(),\n    }\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description="Evaluation d\'un checkpoint Temporal Transformer.")\n    parser.add_argument("--checkpoint", type=str, required=True, help="Chemin du checkpoint .pt.")\n    parser.add_argument("--output-dir", type=str, default="eval_outputs", help="Dossier de sortie des artefacts.")\n    parser.add_argument("--split", type=str, choices=["train", "val", "test"], default="test")\n    parser.add_argument("--device", type=str, default="auto")\n    parser.add_argument("--batch-size", type=int, default=128)\n\n    parser.add_argument("--config-json", type=str, default=None, help="Fichier config.json optionnel du run.")\n    parser.add_argument("--csv-path", type=str, default=None, help="Chemin du CSV d\'entree (si pas de NPZ).")\n    parser.add_argument("--prepared-npz", type=str, default=None, help="Chemin du NPZ prepare.")\n    parser.add_argument("--min-obs", type=int, default=None)\n    parser.add_argument("--split-method", type=str, choices=["parcel", "tile"], default=None)\n    parser.add_argument("--time-grid-frequency", type=str, default=None)\n    parser.add_argument("--index-filter", type=str, default=None)\n    parser.add_argument("--min-px-count", type=int, default=None)\n    parser.add_argument("--max-cloud-scene", type=float, default=None)\n\n    parser.add_argument("--d-model", type=int, default=None)\n    parser.add_argument("--n-heads", type=int, default=None)\n    parser.add_argument("--n-layers", type=int, default=None)\n    parser.add_argument("--ff-dim", type=int, default=None)\n    parser.add_argument("--dropout", type=float, default=None)\n    parser.add_argument("--pooling", type=str, choices=["cls", "mean"], default=None)\n    return parser.parse_args()\n\n\ndef main() -> None:\n    args = parse_args()\n    output_dir = Path(args.output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    logger = setup_logger(output_dir / "evaluate.log")\n\n    cfg = get_default_config()\n    if args.config_json is not None:\n        with Path(args.config_json).open("r", encoding="utf-8") as f:\n            payload = json.load(f)\n        cfg.data = cfg.data.__class__(**payload.get("data", {}))\n        cfg.model = cfg.model.__class__(**payload.get("model", {}))\n        cfg.train = cfg.train.__class__(**payload.get("train", {}))\n\n    if args.csv_path is not None:\n        cfg.data.csv_path = args.csv_path\n        cfg.data.prepared_npz_path = None\n    if args.prepared_npz is not None:\n        cfg.data.prepared_npz_path = args.prepared_npz\n    if args.min_obs is not None:\n        cfg.data.min_obs_per_parcel = args.min_obs\n    if args.split_method is not None:\n        cfg.data.split_method = args.split_method\n    if args.time_grid_frequency is not None:\n        cfg.data.time_grid_frequency = args.time_grid_frequency\n    if args.index_filter is not None:\n        cfg.data.index_filter = [x.strip() for x in args.index_filter.split(",") if x.strip()]\n    if args.min_px_count is not None:\n        cfg.data.min_px_count = args.min_px_count\n    if args.max_cloud_scene is not None:\n        cfg.data.max_cloud_scene = args.max_cloud_scene\n\n    if args.d_model is not None:\n        cfg.model.d_model = args.d_model\n    if args.n_heads is not None:\n        cfg.model.n_heads = args.n_heads\n    if args.n_layers is not None:\n        cfg.model.n_layers = args.n_layers\n    if args.ff_dim is not None:\n        cfg.model.dim_feedforward = args.ff_dim\n    if args.dropout is not None:\n        cfg.model.dropout = args.dropout\n    if args.pooling is not None:\n        cfg.model.pooling = args.pooling\n\n    if args.csv_path is None and args.prepared_npz is None and cfg.data.prepared_npz_path is None:\n        raise ValueError("Fournis --csv-path ou --prepared-npz (ou prepared_npz_path dans la config).")\n\n    prepared = prepare_dataset(cfg.data)\n    if args.split not in prepared.splits:\n        raise ValueError(f"Le split \'{args.split}\' est introuvable dans le dataset prepare.")\n    if prepared.splits[args.split].shape[0] == 0:\n        raise ValueError(f"Le split \'{args.split}\' est vide.")\n    if cfg.train.standardize_features:\n        standardize_prepared_features(\n            prepared=prepared,\n            train_indices=prepared.splits["train"],\n            eps=cfg.train.standardize_eps,\n        )\n        logger.info(\n            "Standardisation des features basee sur le train uniquement (eps=%.1e) avant evaluation.",\n            cfg.train.standardize_eps,\n        )\n\n    device = resolve_device(args.device)\n    dataloaders = build_dataloaders(\n        prepared=prepared,\n        batch_size=args.batch_size,\n        num_workers=cfg.train.num_workers,\n        pin_memory=(device.type == "cuda"),\n    )\n\n    try:\n        checkpoint = torch.load(args.checkpoint, map_location=device, weights_only=False)\n    except TypeError:\n        checkpoint = torch.load(args.checkpoint, map_location=device)\n    metadata = checkpoint.get("metadata", {})\n    cfg.model.reliability_aware = checkpoint_uses_reliability(checkpoint)\n    num_group_classes = int(metadata.get("num_group_classes", 0) or 0)\n    model = TemporalTransformerClassifier(\n        input_dim=prepared.num_features,\n        num_classes=prepared.num_classes,\n        cfg=cfg.model,\n        num_group_classes=num_group_classes,\n    ).to(device)\n    model.load_state_dict(checkpoint["model_state_dict"])\n    configure_hierarchical_constraint_from_metadata(model=model, metadata=metadata, logger=logger)\n    logger.info("Checkpoint charge: %s", args.checkpoint)\n\n    metrics = evaluate_split(\n        model=model,\n        dataloader=dataloaders[args.split],\n        device=device,\n        label_names=prepared.label_names,\n        pooling=cfg.model.pooling,\n        return_attention=True,\n    )\n\n    split_name_fr = {"train": "entrainement", "val": "validation", "test": "test"}.get(args.split, args.split)\n\n    save_json(_serialize_metrics(metrics), output_dir / f"{split_name_fr}_metriques.json")\n    save_json(_serialize_metrics(metrics), output_dir / f"{args.split}_metrics.json")\n\n    with (output_dir / f"{split_name_fr}_rapport_classification.txt").open("w", encoding="utf-8") as f:\n        f.write(metrics["classification_report_text"])\n    with (output_dir / f"{args.split}_classification_report.txt").open("w", encoding="utf-8") as f:\n        f.write(metrics["classification_report_text"])\n    plot_confusion_matrix(\n        confusion=metrics["confusion_matrix"],\n        label_names=prepared.label_names,\n        output_path=output_dir / f"{split_name_fr}_matrice_confusion.png",\n        normalize=False,\n    )\n    plot_confusion_matrix(\n        confusion=metrics["confusion_matrix"],\n        label_names=prepared.label_names,\n        output_path=output_dir / f"{split_name_fr}_matrice_confusion_normalisee.png",\n        normalize=True,\n    )\n    # Compatibilite descendante\n    plot_confusion_matrix(\n        confusion=metrics["confusion_matrix"],\n        label_names=prepared.label_names,\n        output_path=output_dir / f"{args.split}_confusion_matrix.png",\n        normalize=False,\n    )\n    plot_confusion_matrix(\n        confusion=metrics["confusion_matrix"],\n        label_names=prepared.label_names,\n        output_path=output_dir / f"{args.split}_confusion_matrix_normalized.png",\n        normalize=True,\n    )\n\n    predictions = metrics["predictions"]\n    if "temporal_attention" in predictions:\n        att_df = pd.DataFrame(\n            {\n                "date": pd.to_datetime(prepared.time_grid),\n                "day_of_year": pd.to_datetime(prepared.time_grid).dayofyear,\n                "mean_attention": predictions["temporal_attention"],\n            }\n        )\n        att_df.to_csv(output_dir / f"{split_name_fr}_attention_temporelle.csv", index=False)\n        att_df.to_csv(output_dir / f"{args.split}_temporal_attention.csv", index=False)\n\n    logger.info(\n        "%s | exactitude=%.4f precision_macro=%.4f rappel_macro=%.4f f1_macro=%.4f f1_pondere=%.4f",\n        split_name_fr.upper(),\n        metrics["accuracy"],\n        metrics["precision_macro"],\n        metrics["recall_macro"],\n        metrics["f1_macro"],\n        metrics["f1_weighted"],\n    )\n\n\nif __name__ == "__main__":\n    main()\n'
embedded_sources['evaluate_ensemble'] = 'from __future__ import annotations\n\nimport argparse\nimport glob\nimport json\nimport math\nfrom dataclasses import replace\nfrom pathlib import Path\nfrom typing import Any, Optional\n\nimport numpy as np\nimport pandas as pd\nimport torch\n\nfrom config import get_default_config\nfrom data import build_dataloaders, prepare_dataset, standardize_prepared_features\nfrom evaluate import analyze_errors_by_class, compute_classification_metrics, plot_confusion_matrix\nfrom model import TemporalTransformerClassifier\nfrom utils import resolve_device, save_json, setup_logger\n\n\ndef configure_hierarchical_constraint_from_metadata(\n    model: TemporalTransformerClassifier,\n    metadata: dict[str, Any],\n    logger,\n) -> None:\n    enabled = bool(metadata.get("hierarchical_constraint", False))\n    compat = metadata.get("class_group_compat")\n    if not enabled or compat is None:\n        return\n\n    try:\n        compat_tensor = torch.tensor(compat, dtype=torch.float32, device=next(model.parameters()).device)\n        model.configure_hierarchical_constraint(\n            class_group_compat=compat_tensor,\n            weight=float(metadata.get("hierarchical_constraint_weight", 1.0)),\n            eps=float(metadata.get("hierarchical_constraint_eps", 1.0e-6)),\n            enabled=True,\n        )\n        logger.info(\n            "Applied hierarchical constraint for checkpoint model (weight=%.3f).",\n            float(metadata.get("hierarchical_constraint_weight", 1.0)),\n        )\n    except Exception as exc:\n        logger.warning("Could not apply hierarchical constraint from metadata: %s", exc)\n\n\ndef checkpoint_uses_reliability(checkpoint: dict[str, Any]) -> bool:\n    metadata = checkpoint.get("metadata", {})\n    if isinstance(metadata, dict) and "reliability_aware" in metadata:\n        return bool(metadata.get("reliability_aware", False))\n\n    state_dict = checkpoint.get("model_state_dict", {})\n    if not isinstance(state_dict, dict):\n        return False\n\n    for raw_key in state_dict.keys():\n        key = str(raw_key)\n        if key.startswith("module."):\n            key = key[len("module.") :]\n        if key.startswith("backbone."):\n            key = key[len("backbone.") :]\n        if key.startswith("reliability_proj.") or key == "reliability_gate_logit":\n            return True\n    return False\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description="Evaluate an ensemble of Temporal Transformer checkpoints.")\n    parser.add_argument(\n        "--checkpoints",\n        type=str,\n        nargs="*",\n        default=None,\n        help="One or more checkpoint paths (.pt).",\n    )\n    parser.add_argument(\n        "--checkpoint-glob",\n        type=str,\n        default=None,\n        help="Glob pattern for checkpoints, e.g. outputs_transformer/seeds_*/**/best_model.pt",\n    )\n    parser.add_argument("--config-json", type=str, default=None, help="Optional config.json override.")\n    parser.add_argument("--output-dir", type=str, default="eval_outputs_ensemble", help="Output directory.")\n    parser.add_argument("--split", type=str, choices=["train", "val", "test"], default="test")\n    parser.add_argument("--device", type=str, default="auto")\n    parser.add_argument("--batch-size", type=int, default=128)\n    parser.add_argument(\n        "--ensemble-weighting",\n        type=str,\n        choices=["uniform", "val_macro_f1"],\n        default="uniform",\n        help="How to weight checkpoints in the ensemble.",\n    )\n    parser.add_argument(\n        "--weight-power",\n        type=float,\n        default=1.0,\n        help="Exponent applied to non-uniform weights before normalization.",\n    )\n\n    parser.add_argument("--csv-path", type=str, default=None, help="CSV input path (if not using NPZ).")\n    parser.add_argument("--prepared-npz", type=str, default=None, help="Prepared NPZ input path.")\n    parser.add_argument("--min-obs", type=int, default=None)\n    parser.add_argument("--split-method", type=str, choices=["parcel", "tile"], default=None)\n    parser.add_argument("--time-grid-frequency", type=str, default=None)\n    parser.add_argument("--index-filter", type=str, default=None)\n    parser.add_argument("--min-px-count", type=int, default=None)\n    parser.add_argument("--max-cloud-scene", type=float, default=None)\n\n    parser.add_argument("--d-model", type=int, default=None)\n    parser.add_argument("--n-heads", type=int, default=None)\n    parser.add_argument("--n-layers", type=int, default=None)\n    parser.add_argument("--ff-dim", type=int, default=None)\n    parser.add_argument("--dropout", type=float, default=None)\n    parser.add_argument("--pooling", type=str, choices=["cls", "mean"], default=None)\n    return parser.parse_args()\n\n\ndef resolve_checkpoint_paths(args: argparse.Namespace) -> list[Path]:\n    raw_paths: list[str] = []\n    if args.checkpoints:\n        raw_paths.extend(args.checkpoints)\n    if args.checkpoint_glob:\n        raw_paths.extend(glob.glob(args.checkpoint_glob, recursive=True))\n\n    if not raw_paths:\n        raise ValueError("Provide at least one checkpoint via --checkpoints or --checkpoint-glob.")\n\n    deduped: list[Path] = []\n    seen: set[str] = set()\n    for p in raw_paths:\n        resolved = str(Path(p).resolve())\n        if resolved in seen:\n            continue\n        seen.add(resolved)\n        deduped.append(Path(p))\n\n    for p in deduped:\n        if not p.exists():\n            raise FileNotFoundError(f"Checkpoint not found: {p}")\n    return deduped\n\n\ndef load_config(args: argparse.Namespace, checkpoints: list[Path]) -> tuple[Any, Optional[Path]]:\n    cfg = get_default_config()\n\n    config_path: Optional[Path] = None\n    if args.config_json is not None:\n        config_path = Path(args.config_json)\n    else:\n        auto_path = checkpoints[0].parent / "config.json"\n        if auto_path.exists():\n            config_path = auto_path\n\n    if config_path is not None:\n        with config_path.open("r", encoding="utf-8") as f:\n            payload = json.load(f)\n        cfg.data = cfg.data.__class__(**payload.get("data", {}))\n        cfg.model = cfg.model.__class__(**payload.get("model", {}))\n        cfg.train = cfg.train.__class__(**payload.get("train", {}))\n\n    if args.csv_path is not None:\n        cfg.data.csv_path = args.csv_path\n        cfg.data.prepared_npz_path = None\n    if args.prepared_npz is not None:\n        cfg.data.prepared_npz_path = args.prepared_npz\n    if args.min_obs is not None:\n        cfg.data.min_obs_per_parcel = args.min_obs\n    if args.split_method is not None:\n        cfg.data.split_method = args.split_method\n    if args.time_grid_frequency is not None:\n        cfg.data.time_grid_frequency = args.time_grid_frequency\n    if args.index_filter is not None:\n        cfg.data.index_filter = [x.strip() for x in args.index_filter.split(",") if x.strip()]\n    if args.min_px_count is not None:\n        cfg.data.min_px_count = args.min_px_count\n    if args.max_cloud_scene is not None:\n        cfg.data.max_cloud_scene = args.max_cloud_scene\n\n    if args.d_model is not None:\n        cfg.model.d_model = args.d_model\n    if args.n_heads is not None:\n        cfg.model.n_heads = args.n_heads\n    if args.n_layers is not None:\n        cfg.model.n_layers = args.n_layers\n    if args.ff_dim is not None:\n        cfg.model.dim_feedforward = args.ff_dim\n    if args.dropout is not None:\n        cfg.model.dropout = args.dropout\n    if args.pooling is not None:\n        cfg.model.pooling = args.pooling\n\n    if args.csv_path is None and args.prepared_npz is None and cfg.data.prepared_npz_path is None:\n        raise ValueError("Provide --csv-path or --prepared-npz (or ensure config includes prepared_npz_path).")\n\n    return cfg, config_path\n\n\ndef _safe_float(value: Any) -> Optional[float]:\n    if value is None:\n        return None\n    try:\n        x = float(value)\n    except (TypeError, ValueError):\n        return None\n    if not math.isfinite(x):\n        return None\n    return x\n\n\ndef resolve_model_scores(\n    checkpoint_paths: list[Path],\n    checkpoints: list[dict[str, Any]],\n    logger,\n) -> list[float]:\n    scores: list[float] = []\n    for ckpt_path, checkpoint in zip(checkpoint_paths, checkpoints):\n        score: Optional[float] = None\n        val_metrics_path = ckpt_path.parent / "val_metrics.json"\n        if val_metrics_path.exists():\n            try:\n                with val_metrics_path.open("r", encoding="utf-8") as f:\n                    payload = json.load(f)\n                score = _safe_float(payload.get("f1_macro"))\n            except Exception as exc:\n                logger.warning("Could not read %s: %s", val_metrics_path, exc)\n        if score is None:\n            score = _safe_float(checkpoint.get("best_metric"))\n        if score is None:\n            logger.warning("No usable validation score found for %s; defaulting to 0.", ckpt_path)\n            score = 0.0\n        scores.append(max(0.0, score))\n    return scores\n\n\ndef build_model_weights(\n    checkpoint_paths: list[Path],\n    checkpoints: list[dict[str, Any]],\n    weighting: str,\n    weight_power: float,\n    logger,\n) -> tuple[np.ndarray, list[float]]:\n    n_models = len(checkpoint_paths)\n    if n_models == 0:\n        raise ValueError("No checkpoints available to build ensemble weights.")\n\n    if weighting == "uniform":\n        return np.full((n_models,), 1.0 / float(n_models), dtype=np.float32), [1.0] * n_models\n\n    raw_scores = resolve_model_scores(checkpoint_paths=checkpoint_paths, checkpoints=checkpoints, logger=logger)\n    scores = np.asarray(raw_scores, dtype=np.float64)\n    scores = np.clip(scores, a_min=0.0, a_max=None)\n\n    if weight_power <= 0:\n        raise ValueError("--weight-power must be > 0.")\n    if weight_power != 1.0:\n        scores = np.power(scores, weight_power)\n\n    total = float(scores.sum())\n    if total <= 0:\n        logger.warning("All model scores are zero; falling back to uniform ensemble weights.")\n        return np.full((n_models,), 1.0 / float(n_models), dtype=np.float32), raw_scores\n\n    weights = (scores / total).astype(np.float32)\n    return weights, raw_scores\n\n\n@torch.no_grad()\ndef predict_ensemble(\n    models: list[torch.nn.Module],\n    model_weights: np.ndarray,\n    dataloader: torch.utils.data.DataLoader,\n    device: torch.device,\n) -> dict[str, np.ndarray]:\n    if len(models) == 0:\n        raise ValueError("No models were loaded for ensemble prediction.")\n    if model_weights.shape[0] != len(models):\n        raise ValueError("model_weights size must match number of models.")\n\n    for model in models:\n        model.eval()\n\n    all_logits: list[np.ndarray] = []\n    all_labels: list[np.ndarray] = []\n    all_parcel_ids: list[str] = []\n\n    for batch in dataloader:\n        features = batch["features"].to(device=device, dtype=torch.float32)\n        day_of_year = batch["day_of_year"].to(device=device, dtype=torch.float32)\n        observed_mask = batch["observed_mask"].to(device=device, dtype=torch.bool)\n        quality_features = batch["quality_features"].to(device=device, dtype=torch.float32)\n        labels = batch["label"].to(device=device, dtype=torch.long)\n\n        logits_sum: Optional[torch.Tensor] = None\n        for i, model in enumerate(models):\n            outputs = model(\n                features=features,\n                day_of_year=day_of_year,\n                observed_mask=observed_mask,\n                quality_features=quality_features,\n                return_attention=False,\n            )\n            logits = outputs["logits"] * float(model_weights[i])\n            logits_sum = logits if logits_sum is None else (logits_sum + logits)\n\n        assert logits_sum is not None\n        all_logits.append(logits_sum.cpu().numpy())\n        all_labels.append(labels.cpu().numpy())\n        all_parcel_ids.extend(batch["parcel_id"])\n\n    logits_np = np.concatenate(all_logits, axis=0) if all_logits else np.empty((0, 0), dtype=np.float32)\n    labels_np = np.concatenate(all_labels, axis=0) if all_labels else np.empty((0,), dtype=np.int64)\n    preds_np = logits_np.argmax(axis=1) if logits_np.size else np.empty((0,), dtype=np.int64)\n    return {\n        "logits": logits_np,\n        "labels": labels_np,\n        "preds": preds_np,\n        "parcel_ids": np.asarray(all_parcel_ids, dtype=object),\n    }\n\n\ndef main() -> None:\n    args = parse_args()\n    checkpoint_paths = resolve_checkpoint_paths(args)\n\n    output_dir = Path(args.output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    logger = setup_logger(output_dir / "evaluate_ensemble.log")\n\n    cfg, config_path = load_config(args, checkpoint_paths)\n    if config_path is not None:\n        logger.info("Loaded config from %s", config_path)\n    logger.info("Using %d checkpoints in ensemble.", len(checkpoint_paths))\n    logger.info("Checkpoints: %s", [str(p) for p in checkpoint_paths])\n    save_json(\n        {\n            "checkpoints": [str(p) for p in checkpoint_paths],\n            "split": args.split,\n            "device": args.device,\n            "batch_size": args.batch_size,\n            "ensemble_weighting": args.ensemble_weighting,\n            "weight_power": args.weight_power,\n            "config": cfg.to_dict(),\n        },\n        output_dir / "ensemble_config.json",\n    )\n\n    prepared = prepare_dataset(cfg.data)\n    if args.split not in prepared.splits:\n        raise ValueError(f"Split \'{args.split}\' not found in prepared dataset.")\n    if prepared.splits[args.split].shape[0] == 0:\n        raise ValueError(f"Split \'{args.split}\' is empty.")\n\n    if cfg.train.standardize_features:\n        standardize_prepared_features(\n            prepared=prepared,\n            train_indices=prepared.splits["train"],\n            eps=cfg.train.standardize_eps,\n        )\n        logger.info(\n            "Applied train-only feature standardization (eps=%.1e) before ensemble evaluation.",\n            cfg.train.standardize_eps,\n        )\n\n    device = resolve_device(args.device)\n    dataloaders = build_dataloaders(\n        prepared=prepared,\n        batch_size=args.batch_size,\n        num_workers=cfg.train.num_workers,\n        pin_memory=(device.type == "cuda"),\n    )\n    dataloader = dataloaders[args.split]\n\n    models: list[torch.nn.Module] = []\n    loaded_checkpoints: list[dict[str, Any]] = []\n    for ckpt_path in checkpoint_paths:\n        try:\n            checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)\n        except TypeError:\n            checkpoint = torch.load(ckpt_path, map_location=device)\n\n        metadata = checkpoint.get("metadata", {})\n        uses_reliability = checkpoint_uses_reliability(checkpoint)\n        num_group_classes = int(metadata.get("num_group_classes", 0) or 0)\n        model_cfg = replace(cfg.model, reliability_aware=uses_reliability)\n        model = TemporalTransformerClassifier(\n            input_dim=prepared.num_features,\n            num_classes=prepared.num_classes,\n            cfg=model_cfg,\n            num_group_classes=num_group_classes,\n        ).to(device)\n        label_names_meta = metadata.get("label_names")\n        if label_names_meta is not None and list(label_names_meta) != prepared.label_names:\n            raise ValueError(\n                f"Label names mismatch for checkpoint {ckpt_path}. "\n                "Check that all ensemble models were trained on the same class set."\n            )\n\n        model.load_state_dict(checkpoint["model_state_dict"])\n        configure_hierarchical_constraint_from_metadata(model=model, metadata=metadata, logger=logger)\n        models.append(model)\n        loaded_checkpoints.append(checkpoint)\n\n    model_weights, raw_scores = build_model_weights(\n        checkpoint_paths=checkpoint_paths,\n        checkpoints=loaded_checkpoints,\n        weighting=args.ensemble_weighting,\n        weight_power=args.weight_power,\n        logger=logger,\n    )\n    for ckpt_path, raw_score, w in zip(checkpoint_paths, raw_scores, model_weights):\n        logger.info(\n            "Ensemble weight | checkpoint=%s | raw_score=%.6f | weight=%.6f",\n            ckpt_path,\n            float(raw_score),\n            float(w),\n        )\n\n    predictions = predict_ensemble(\n        models=models,\n        model_weights=model_weights,\n        dataloader=dataloader,\n        device=device,\n    )\n    metrics = compute_classification_metrics(\n        y_true=predictions["labels"],\n        y_pred=predictions["preds"],\n        label_names=prepared.label_names,\n    )\n    metrics["error_analysis"] = analyze_errors_by_class(\n        confusion=metrics["confusion_matrix"],\n        label_names=prepared.label_names,\n    )\n\n    metrics_payload = {\n        "accuracy": metrics["accuracy"],\n        "recall_macro": metrics["recall_macro"],\n        "recall_weighted": metrics["recall_weighted"],\n        "f1_macro": metrics["f1_macro"],\n        "f1_weighted": metrics["f1_weighted"],\n        "classification_report": metrics["classification_report_dict"],\n        "error_analysis": metrics["error_analysis"],\n        "confusion_matrix": metrics["confusion_matrix"].tolist(),\n        "n_models": len(models),\n        "checkpoints": [str(p) for p in checkpoint_paths],\n        "ensemble_weighting": args.ensemble_weighting,\n        "weight_power": args.weight_power,\n        "model_weights": [float(x) for x in model_weights.tolist()],\n        "model_raw_scores": [float(x) for x in raw_scores],\n    }\n    save_json(metrics_payload, output_dir / f"{args.split}_ensemble_metrics.json")\n    with (output_dir / f"{args.split}_ensemble_classification_report.txt").open("w", encoding="utf-8") as f:\n        f.write(metrics["classification_report_text"])\n\n    plot_confusion_matrix(\n        confusion=metrics["confusion_matrix"],\n        label_names=prepared.label_names,\n        output_path=output_dir / f"{args.split}_ensemble_confusion_matrix.png",\n        normalize=False,\n    )\n    plot_confusion_matrix(\n        confusion=metrics["confusion_matrix"],\n        label_names=prepared.label_names,\n        output_path=output_dir / f"{args.split}_ensemble_confusion_matrix_normalized.png",\n        normalize=True,\n    )\n\n    true_labels = [prepared.label_names[i] for i in predictions["labels"]]\n    pred_labels = [prepared.label_names[i] for i in predictions["preds"]]\n    pred_df = pd.DataFrame(\n        {\n            "parcel_id": predictions["parcel_ids"],\n            "y_true_idx": predictions["labels"],\n            "y_pred_idx": predictions["preds"],\n            "y_true_label": true_labels,\n            "y_pred_label": pred_labels,\n        }\n    )\n    pred_df.to_csv(output_dir / f"{args.split}_ensemble_predictions.csv", index=False)\n\n    logger.info(\n        "ENSEMBLE %s | n_models=%d | accuracy=%.4f recall_macro=%.4f macro_f1=%.4f weighted_f1=%.4f",\n        args.split.upper(),\n        len(models),\n        metrics["accuracy"],\n        metrics["recall_macro"],\n        metrics["f1_macro"],\n        metrics["f1_weighted"],\n    )\n    logger.info("Artifacts saved in %s", output_dir)\n\n\nif __name__ == "__main__":\n    main()\n'
embedded_sources['pretrain_ssl'] = 'from __future__ import annotations\n\nimport argparse\nimport math\nfrom pathlib import Path\nfrom typing import Optional\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import Tensor, nn\nfrom tqdm import tqdm\n\nfrom config import ExperimentConfig, get_default_config\nfrom data import build_dataloaders, prepare_dataset, standardize_prepared_features\nfrom model import TemporalTransformerClassifier\nfrom utils import EarlyStopping, create_run_dir, resolve_device, save_checkpoint, save_json, set_seed, setup_logger\n\n\nclass SSLPretrainModel(nn.Module):\n    def __init__(self, backbone: TemporalTransformerClassifier) -> None:\n        super().__init__()\n        self.backbone = backbone\n        d_model = backbone.feature_proj.out_features\n        self.reconstruction_head = nn.Sequential(\n            nn.LayerNorm(d_model),\n            nn.Linear(d_model, backbone.input_dim),\n        )\n\n    def forward(\n        self,\n        features: Tensor,\n        day_of_year: Tensor,\n        observed_mask: Tensor,\n        quality_features: Tensor,\n    ) -> Tensor:\n        encoded = self.backbone.encode(\n            features=features,\n            day_of_year=day_of_year,\n            observed_mask=observed_mask,\n            quality_features=quality_features,\n            return_attention=False,\n        )\n        temporal_tokens = encoded["temporal_tokens"]\n        return self.reconstruction_head(temporal_tokens)\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description="Self-supervised masked pretraining for parcel temporal transformer.")\n    parser.add_argument("--csv-path", type=str, default=None, help="Path to long CSV input.")\n    parser.add_argument("--prepared-npz", type=str, default=None, help="Prepared NPZ input path.")\n    parser.add_argument("--save-prepared-npz", type=str, default=None, help="Optional output NPZ path.")\n    parser.add_argument("--output-dir", type=str, default="outputs_transformer/ssl_pretrain")\n\n    parser.add_argument("--split-method", type=str, choices=["parcel", "tile"], default=None)\n    parser.add_argument("--time-grid-frequency", type=str, default=None)\n    parser.add_argument("--index-filter", type=str, default=None)\n    parser.add_argument("--min-obs", type=int, default=None)\n    parser.add_argument("--max-cloud-scene", type=float, default=None)\n    parser.add_argument("--min-px-count", type=int, default=None)\n\n    parser.add_argument("--d-model", type=int, default=None)\n    parser.add_argument("--n-heads", type=int, default=None)\n    parser.add_argument("--n-layers", type=int, default=None)\n    parser.add_argument("--ff-dim", type=int, default=None)\n    parser.add_argument("--dropout", type=float, default=None)\n    parser.add_argument("--pooling", type=str, choices=["cls", "mean"], default=None)\n    parser.add_argument(\n        "--reliability-aware",\n        dest="reliability_aware",\n        action="store_true",\n        default=None,\n        help="Enable reliability-aware encoder during SSL pretraining.",\n    )\n    parser.add_argument(\n        "--no-reliability-aware",\n        dest="reliability_aware",\n        action="store_false",\n        default=None,\n    )\n\n    parser.add_argument("--epochs", type=int, default=30)\n    parser.add_argument("--batch-size", type=int, default=64)\n    parser.add_argument("--lr", type=float, default=7e-4)\n    parser.add_argument("--weight-decay", type=float, default=1e-2)\n    parser.add_argument("--num-workers", type=int, default=0)\n    parser.add_argument("--scheduler", type=str, choices=["none", "plateau", "cosine"], default="plateau")\n    parser.add_argument("--scheduler-patience", type=int, default=4)\n    parser.add_argument("--scheduler-factor", type=float, default=0.5)\n    parser.add_argument("--min-learning-rate", type=float, default=1e-6)\n    parser.add_argument("--early-stopping-patience", type=int, default=8)\n    parser.add_argument("--gradient-clip-norm", type=float, default=1.0)\n    parser.add_argument("--mask-ratio", type=float, default=0.25, help="Fraction of observed timestamps masked.")\n    parser.add_argument("--ssl-loss", type=str, choices=["mse", "l1"], default="mse")\n    parser.add_argument(\n        "--standardize-features",\n        dest="standardize_features",\n        action="store_true",\n        default=True,\n        help="Apply train-only feature standardization before SSL training.",\n    )\n    parser.add_argument(\n        "--no-standardize-features",\n        dest="standardize_features",\n        action="store_false",\n    )\n    parser.add_argument("--device", type=str, default="auto")\n    parser.add_argument("--seed", type=int, default=42)\n    return parser.parse_args()\n\n\ndef apply_args_to_config(args: argparse.Namespace, cfg: ExperimentConfig) -> ExperimentConfig:\n    if args.csv_path is not None:\n        cfg.data.csv_path = args.csv_path\n    if args.prepared_npz is not None:\n        cfg.data.prepared_npz_path = args.prepared_npz\n    if args.save_prepared_npz is not None:\n        cfg.data.save_prepared_npz_path = args.save_prepared_npz\n    if args.output_dir is not None:\n        cfg.data.output_dir = args.output_dir\n\n    if args.split_method is not None:\n        cfg.data.split_method = args.split_method\n    if args.time_grid_frequency is not None:\n        cfg.data.time_grid_frequency = args.time_grid_frequency\n    if args.index_filter is not None:\n        cfg.data.index_filter = [x.strip() for x in args.index_filter.split(",") if x.strip()]\n    if args.min_obs is not None:\n        cfg.data.min_obs_per_parcel = args.min_obs\n    if args.max_cloud_scene is not None:\n        cfg.data.max_cloud_scene = args.max_cloud_scene\n    if args.min_px_count is not None:\n        cfg.data.min_px_count = args.min_px_count\n\n    if args.d_model is not None:\n        cfg.model.d_model = args.d_model\n    if args.n_heads is not None:\n        cfg.model.n_heads = args.n_heads\n    if args.n_layers is not None:\n        cfg.model.n_layers = args.n_layers\n    if args.ff_dim is not None:\n        cfg.model.dim_feedforward = args.ff_dim\n    if args.dropout is not None:\n        cfg.model.dropout = args.dropout\n    if args.pooling is not None:\n        cfg.model.pooling = args.pooling\n    if args.reliability_aware is not None:\n        cfg.model.reliability_aware = args.reliability_aware\n\n    cfg.train.epochs = args.epochs\n    cfg.train.batch_size = args.batch_size\n    cfg.train.learning_rate = args.lr\n    cfg.train.weight_decay = args.weight_decay\n    cfg.train.num_workers = args.num_workers\n    cfg.train.scheduler = args.scheduler\n    cfg.train.scheduler_patience = args.scheduler_patience\n    cfg.train.scheduler_factor = args.scheduler_factor\n    cfg.train.min_learning_rate = args.min_learning_rate\n    cfg.train.early_stopping_patience = args.early_stopping_patience\n    cfg.train.gradient_clip_norm = args.gradient_clip_norm\n    cfg.train.standardize_features = args.standardize_features\n    cfg.train.seed = args.seed\n    cfg.train.device = args.device\n    return cfg\n\n\ndef build_scheduler(\n    optimizer: torch.optim.Optimizer,\n    cfg: ExperimentConfig,\n):\n    if cfg.train.scheduler == "none":\n        return None\n    if cfg.train.scheduler == "plateau":\n        return torch.optim.lr_scheduler.ReduceLROnPlateau(\n            optimizer,\n            mode="min",\n            factor=cfg.train.scheduler_factor,\n            patience=cfg.train.scheduler_patience,\n            min_lr=cfg.train.min_learning_rate,\n        )\n    if cfg.train.scheduler == "cosine":\n        return torch.optim.lr_scheduler.CosineAnnealingLR(\n            optimizer,\n            T_max=cfg.train.epochs,\n            eta_min=cfg.train.min_learning_rate,\n        )\n    raise ValueError(f"Unsupported scheduler: {cfg.train.scheduler}")\n\n\ndef sample_ssl_mask(observed_mask: Tensor, mask_ratio: float) -> Tensor:\n    if mask_ratio <= 0.0:\n        return torch.zeros_like(observed_mask, dtype=torch.bool)\n    random_scores = torch.rand(observed_mask.shape, device=observed_mask.device)\n    ssl_mask = (random_scores < mask_ratio) & observed_mask\n    needs_one = (ssl_mask.sum(dim=1) == 0) & (observed_mask.sum(dim=1) > 0)\n    sample_ids = torch.where(needs_one)[0]\n    for i in sample_ids.tolist():\n        observed_idx = torch.where(observed_mask[i])[0]\n        if observed_idx.numel() == 0:\n            continue\n        selected = observed_idx[torch.randint(observed_idx.numel(), (1,), device=observed_idx.device)]\n        ssl_mask[i, selected] = True\n    return ssl_mask\n\n\ndef run_ssl_epoch(\n    model: SSLPretrainModel,\n    dataloader: torch.utils.data.DataLoader,\n    device: torch.device,\n    mask_ratio: float,\n    ssl_loss: str,\n    optimizer: Optional[torch.optim.Optimizer] = None,\n    gradient_clip_norm: Optional[float] = None,\n) -> dict[str, float]:\n    is_train = optimizer is not None\n    model.train(mode=is_train)\n\n    total_loss = 0.0\n    total_abs = 0.0\n    total_sq = 0.0\n    total_values = 0\n    total_masked_tokens = 0\n\n    iterator = tqdm(dataloader, leave=False, desc="ssl_train" if is_train else "ssl_eval")\n    for batch in iterator:\n        features = batch["features"].to(device=device, dtype=torch.float32)\n        day_of_year = batch["day_of_year"].to(device=device, dtype=torch.float32)\n        observed_mask = batch["observed_mask"].to(device=device, dtype=torch.bool)\n        quality_features = batch["quality_features"].to(device=device, dtype=torch.float32)\n\n        ssl_mask = sample_ssl_mask(observed_mask, mask_ratio=mask_ratio)\n        masked_features = features.clone()\n        masked_observed = observed_mask.clone()\n        masked_quality = quality_features.clone()\n\n        masked_features[ssl_mask] = 0.0\n        masked_observed[ssl_mask] = False\n        masked_quality[..., 0] = torch.where(\n            masked_observed, masked_quality[..., 0], torch.zeros_like(masked_quality[..., 0])\n        )\n        masked_quality[..., 1] = torch.where(\n            masked_observed, masked_quality[..., 1], torch.ones_like(masked_quality[..., 1])\n        )\n        masked_quality[..., 2] = torch.where(\n            masked_observed, masked_quality[..., 2], torch.zeros_like(masked_quality[..., 2])\n        )\n\n        if is_train:\n            optimizer.zero_grad(set_to_none=True)\n\n        with torch.set_grad_enabled(is_train):\n            reconstructed = model(\n                features=masked_features,\n                day_of_year=day_of_year,\n                observed_mask=masked_observed,\n                quality_features=masked_quality,\n            )\n            target_values = features[ssl_mask]\n            pred_values = reconstructed[ssl_mask]\n            if target_values.numel() == 0:\n                loss = reconstructed.sum() * 0.0\n                batch_abs = 0.0\n                batch_sq = 0.0\n                batch_values = 0\n                masked_tokens = 0\n            else:\n                diff = pred_values - target_values\n                if ssl_loss == "mse":\n                    loss = diff.pow(2).mean()\n                elif ssl_loss == "l1":\n                    loss = diff.abs().mean()\n                else:\n                    raise ValueError(f"Unsupported ssl_loss: {ssl_loss}")\n                batch_abs = float(diff.abs().sum().item())\n                batch_sq = float(diff.pow(2).sum().item())\n                batch_values = int(diff.numel())\n                masked_tokens = int(ssl_mask.sum().item())\n\n            if is_train:\n                loss.backward()\n                if gradient_clip_norm is not None:\n                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=gradient_clip_norm)\n                optimizer.step()\n\n        if batch_values > 0:\n            total_loss += float(loss.item()) * batch_values\n            total_abs += batch_abs\n            total_sq += batch_sq\n            total_values += batch_values\n            total_masked_tokens += masked_tokens\n\n    if total_values == 0:\n        return {\n            "loss": float("nan"),\n            "mae": float("nan"),\n            "rmse": float("nan"),\n            "masked_tokens": 0.0,\n        }\n\n    return {\n        "loss": total_loss / float(total_values),\n        "mae": total_abs / float(total_values),\n        "rmse": math.sqrt(total_sq / float(total_values)),\n        "masked_tokens": float(total_masked_tokens),\n    }\n\n\ndef extract_encoder_state_dict(backbone: TemporalTransformerClassifier) -> dict[str, Tensor]:\n    allowed_prefixes = (\n        "feature_proj.",\n        "doy_encoding.",\n        "layers.",\n        "cls_token",\n        "reliability_proj.",\n        "reliability_gate_logit",\n    )\n    return {\n        k: v.detach().cpu()\n        for k, v in backbone.state_dict().items()\n        if any(k == prefix or k.startswith(prefix) for prefix in allowed_prefixes)\n    }\n\n\ndef main() -> None:\n    args = parse_args()\n    cfg = apply_args_to_config(args, get_default_config())\n\n    if args.csv_path is None and args.prepared_npz is None and cfg.data.prepared_npz_path is None:\n        raise ValueError("Provide --csv-path or --prepared-npz.")\n    if cfg.train.epochs <= 0:\n        raise ValueError("epochs must be > 0.")\n    if cfg.train.learning_rate <= 0:\n        raise ValueError("lr must be > 0.")\n    if cfg.train.batch_size <= 0:\n        raise ValueError("batch-size must be > 0.")\n    if not 0.0 < args.mask_ratio < 1.0:\n        raise ValueError("mask-ratio must be in (0, 1).")\n    if cfg.train.early_stopping_patience <= 0:\n        raise ValueError("early-stopping-patience must be > 0.")\n\n    set_seed(cfg.train.seed)\n    run_dir = create_run_dir(cfg.data.output_dir, prefix="ssl_pretrain")\n    logger = setup_logger(run_dir / "ssl_pretrain.log")\n    logger.info("Run directory: %s", run_dir)\n    logger.info("Configuration loaded.")\n    logger.info("Reliability-aware encoder: %s", "enabled" if cfg.model.reliability_aware else "disabled")\n    save_json(\n        {\n            "config": cfg.to_dict(),\n            "ssl": {\n                "mask_ratio": args.mask_ratio,\n                "loss": args.ssl_loss,\n            },\n        },\n        run_dir / "ssl_config.json",\n    )\n\n    prepared = prepare_dataset(cfg.data)\n    logger.info(\n        "Prepared dataset: N=%d, T=%d, F=%d, classes=%d",\n        prepared.features.shape[0],\n        prepared.seq_len,\n        prepared.num_features,\n        prepared.num_classes,\n    )\n    split_sizes = {k: int(v.shape[0]) for k, v in prepared.splits.items()}\n    logger.info("Split sizes: %s", split_sizes)\n    if split_sizes.get("train", 0) == 0:\n        raise ValueError("Train split is empty.")\n\n    if cfg.train.standardize_features:\n        scaler_stats = standardize_prepared_features(\n            prepared=prepared,\n            train_indices=prepared.splits["train"],\n            eps=cfg.train.standardize_eps,\n        )\n        save_json(\n            {\n                "feature_names": prepared.feature_names,\n                "mean": scaler_stats["mean"].tolist(),\n                "std": scaler_stats["std"].tolist(),\n                "count_per_feature": scaler_stats["count_per_feature"].tolist(),\n            },\n            run_dir / "feature_scaler.json",\n        )\n        logger.info("Applied train-only feature standardization (eps=%.1e).", cfg.train.standardize_eps)\n\n    device = resolve_device(cfg.train.device)\n    logger.info("Device: %s", device)\n    dataloaders = build_dataloaders(\n        prepared=prepared,\n        batch_size=cfg.train.batch_size,\n        num_workers=cfg.train.num_workers,\n        pin_memory=(device.type == "cuda"),\n    )\n\n    backbone = TemporalTransformerClassifier(\n        input_dim=prepared.num_features,\n        num_classes=prepared.num_classes,\n        cfg=cfg.model,\n        num_group_classes=0,\n    )\n    model = SSLPretrainModel(backbone=backbone).to(device)\n\n    optimizer = torch.optim.AdamW(\n        model.parameters(),\n        lr=cfg.train.learning_rate,\n        weight_decay=cfg.train.weight_decay,\n    )\n    scheduler = build_scheduler(optimizer=optimizer, cfg=cfg)\n    early_stopping = (\n        EarlyStopping(patience=cfg.train.early_stopping_patience, mode="min")\n        if split_sizes.get("val", 0) > 0\n        else None\n    )\n\n    best_metric = float("inf")\n    best_model_path = run_dir / "best_ssl_model.pt"\n    best_encoder_path = run_dir / "best_ssl_encoder.pt"\n    history: list[dict[str, float]] = []\n\n    for epoch in range(1, cfg.train.epochs + 1):\n        train_metrics = run_ssl_epoch(\n            model=model,\n            dataloader=dataloaders["train"],\n            device=device,\n            mask_ratio=args.mask_ratio,\n            ssl_loss=args.ssl_loss,\n            optimizer=optimizer,\n            gradient_clip_norm=cfg.train.gradient_clip_norm,\n        )\n\n        if split_sizes.get("val", 0) > 0:\n            val_metrics = run_ssl_epoch(\n                model=model,\n                dataloader=dataloaders["val"],\n                device=device,\n                mask_ratio=args.mask_ratio,\n                ssl_loss=args.ssl_loss,\n                optimizer=None,\n                gradient_clip_norm=None,\n            )\n            monitor_metric = val_metrics["loss"]\n        else:\n            val_metrics = {"loss": float("nan"), "mae": float("nan"), "rmse": float("nan"), "masked_tokens": 0.0}\n            monitor_metric = train_metrics["loss"]\n\n        current_lr = float(optimizer.param_groups[0]["lr"])\n        logger.info(\n            "Epoch %03d | lr=%.6f | train_loss=%.6f train_rmse=%.6f | val_loss=%.6f val_rmse=%.6f",\n            epoch,\n            current_lr,\n            train_metrics["loss"],\n            train_metrics["rmse"],\n            val_metrics["loss"],\n            val_metrics["rmse"],\n        )\n        history.append(\n            {\n                "epoch": float(epoch),\n                "lr": current_lr,\n                "train_loss": train_metrics["loss"],\n                "train_mae": train_metrics["mae"],\n                "train_rmse": train_metrics["rmse"],\n                "train_masked_tokens": train_metrics["masked_tokens"],\n                "val_loss": val_metrics["loss"],\n                "val_mae": val_metrics["mae"],\n                "val_rmse": val_metrics["rmse"],\n                "val_masked_tokens": val_metrics["masked_tokens"],\n            }\n        )\n\n        if monitor_metric < best_metric:\n            best_metric = monitor_metric\n            save_checkpoint(\n                path=best_model_path,\n                model=model,\n                optimizer=optimizer,\n                scheduler=scheduler,\n                epoch=epoch,\n                best_metric=best_metric,\n                metadata={\n                    "task": "ssl_masked_reconstruction",\n                    "mask_ratio": args.mask_ratio,\n                    "ssl_loss": args.ssl_loss,\n                    "feature_names": prepared.feature_names,\n                    "reliability_aware": bool(cfg.model.reliability_aware),\n                    "model_config": {\n                        "d_model": cfg.model.d_model,\n                        "n_heads": cfg.model.n_heads,\n                        "n_layers": cfg.model.n_layers,\n                        "dim_feedforward": cfg.model.dim_feedforward,\n                        "dropout": cfg.model.dropout,\n                        "pooling": cfg.model.pooling,\n                        "reliability_aware": cfg.model.reliability_aware,\n                    },\n                },\n            )\n            torch.save(\n                {\n                    "encoder_state_dict": extract_encoder_state_dict(model.backbone),\n                    "metadata": {\n                        "task": "ssl_masked_reconstruction",\n                        "mask_ratio": args.mask_ratio,\n                        "feature_names": prepared.feature_names,\n                        "reliability_aware": bool(cfg.model.reliability_aware),\n                    },\n                },\n                best_encoder_path,\n            )\n            logger.info("Saved new best SSL checkpoint (loss=%.6f).", best_metric)\n\n        if scheduler is not None:\n            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):\n                scheduler.step(monitor_metric)\n            else:\n                scheduler.step()\n\n        if early_stopping is not None and early_stopping.step(monitor_metric):\n            logger.info("Early stopping triggered at epoch %d.", epoch)\n            break\n\n    pd.DataFrame(history).to_csv(run_dir / "ssl_history.csv", index=False)\n    logger.info("Artifacts saved in %s", run_dir)\n    logger.info("Best SSL model: %s", best_model_path)\n    logger.info("Best encoder checkpoint for fine-tuning: %s", best_encoder_path)\n\n\nif __name__ == "__main__":\n    main()\n'
embedded_sources['train'] = 'from __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\nfrom typing import Optional\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.metrics import accuracy_score, f1_score, recall_score\nfrom torch import Tensor, nn\nfrom torch.nn import functional as F\nfrom torch.utils.data import WeightedRandomSampler\nfrom tqdm import tqdm\n\nfrom config import ExperimentConfig, get_default_config\nfrom data import (\n    TemporalAugmentationConfig,\n    build_dataloaders,\n    prepare_dataset,\n    standardize_prepared_features,\n)\nfrom evaluate import evaluate_split, plot_confusion_matrix\nfrom model import TemporalTransformerClassifier\nfrom utils import (\n    EarlyStopping,\n    compute_class_weights,\n    create_run_dir,\n    resolve_device,\n    save_checkpoint,\n    save_json,\n    set_seed,\n    setup_logger,\n)\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description="Temporal Transformer for parcel crop classification")\n\n    parser.add_argument("--csv-path", type=str, default=None, help="Path to long CSV input.")\n    parser.add_argument("--prepared-npz", type=str, default=None, help="Load prepared dataset from NPZ.")\n    parser.add_argument("--save-prepared-npz", type=str, default=None, help="Optional output NPZ path.")\n    parser.add_argument("--output-dir", type=str, default=None, help="Base output directory.")\n    parser.add_argument(\n        "--label-group-col",\n        type=str,\n        default=None,\n        help="Optional group label column name in CSV (default from config: label_group).",\n    )\n\n    parser.add_argument("--split-method", type=str, choices=["parcel", "tile"], default=None)\n    parser.add_argument("--time-grid-frequency", type=str, default=None, help="e.g. 5D. Default uses observed dates.")\n    parser.add_argument("--index-filter", type=str, default=None, help="Comma-separated list of indices.")\n    parser.add_argument("--min-obs", type=int, default=None, help="Min observed dates per parcel.")\n    parser.add_argument("--max-cloud-scene", type=float, default=None)\n    parser.add_argument("--min-px-count", type=int, default=None)\n\n    parser.add_argument("--d-model", type=int, default=None)\n    parser.add_argument("--n-heads", type=int, default=None)\n    parser.add_argument("--n-layers", type=int, default=None)\n    parser.add_argument("--ff-dim", type=int, default=None)\n    parser.add_argument("--dropout", type=float, default=None)\n    parser.add_argument("--pooling", type=str, choices=["cls", "mean"], default=None)\n    parser.add_argument(\n        "--reliability-aware",\n        dest="reliability_aware",\n        action="store_true",\n        default=None,\n        help="Inject reliability signals (observed mask / cloud / px_count) into temporal encoding and pooling.",\n    )\n    parser.add_argument(\n        "--no-reliability-aware",\n        dest="reliability_aware",\n        action="store_false",\n        default=None,\n    )\n    parser.add_argument(\n        "--pretrained-encoder-checkpoint",\n        type=str,\n        default=None,\n        help="Optional SSL checkpoint (.pt) used to initialize backbone encoder weights before supervised training.",\n    )\n\n    parser.add_argument("--epochs", type=int, default=None)\n    parser.add_argument("--batch-size", type=int, default=None)\n    parser.add_argument("--lr", type=float, default=None)\n    parser.add_argument("--weight-decay", type=float, default=None)\n    parser.add_argument(\n        "--standardize-features",\n        dest="standardize_features",\n        action="store_true",\n        default=None,\n        help="Apply train-only feature standardization using observed mask.",\n    )\n    parser.add_argument(\n        "--no-standardize-features",\n        dest="standardize_features",\n        action="store_false",\n        default=None,\n    )\n    parser.add_argument(\n        "--loss-type",\n        type=str,\n        choices=["cross_entropy", "focal", "balanced_softmax", "logit_adjusted"],\n        default=None,\n    )\n    parser.add_argument("--focal-gamma", type=float, default=None)\n    parser.add_argument(\n        "--logit-adjust-tau",\n        type=float,\n        default=None,\n        help="Tau for logit-adjusted cross entropy (used when --loss-type logit_adjusted).",\n    )\n    parser.add_argument(\n        "--use-group-task",\n        dest="use_group_task",\n        action="store_true",\n        default=None,\n        help="Enable auxiliary group classification head (requires label_group column).",\n    )\n    parser.add_argument(\n        "--no-use-group-task",\n        dest="use_group_task",\n        action="store_false",\n        default=None,\n    )\n    parser.add_argument(\n        "--group-loss-weight",\n        type=float,\n        default=None,\n        help="Weight for auxiliary group loss when --use-group-task is enabled.",\n    )\n    parser.add_argument(\n        "--hierarchical-constraint",\n        dest="hierarchical_constraint",\n        action="store_true",\n        default=None,\n        help="Enable hierarchical constrained decoding (group probabilities constrain class logits).",\n    )\n    parser.add_argument(\n        "--no-hierarchical-constraint",\n        dest="hierarchical_constraint",\n        action="store_false",\n        default=None,\n    )\n    parser.add_argument(\n        "--hierarchical-constraint-weight",\n        type=float,\n        default=None,\n        help="Lambda weight for hierarchical log-compatibility added to class logits.",\n    )\n    parser.add_argument(\n        "--hierarchical-constraint-eps",\n        type=float,\n        default=None,\n        help="Numerical epsilon for hierarchical log-compatibility term.",\n    )\n    parser.add_argument("--scheduler", type=str, choices=["none", "plateau", "cosine"], default=None)\n    parser.add_argument("--class-weighting", action="store_true")\n    parser.add_argument("--no-class-weighting", action="store_true")\n    parser.add_argument("--class-weight-power", type=float, default=None)\n    parser.add_argument(\n        "--weighted-sampler",\n        dest="weighted_sampler",\n        action="store_true",\n        default=None,\n        help="Use WeightedRandomSampler for the train DataLoader.",\n    )\n    parser.add_argument(\n        "--no-weighted-sampler",\n        dest="weighted_sampler",\n        action="store_false",\n        default=None,\n    )\n    parser.add_argument("--sampler-power", type=float, default=None)\n    parser.add_argument(\n        "--temporal-augmentation",\n        dest="temporal_augmentation",\n        action="store_true",\n        default=None,\n        help="Enable light temporal augmentation on train batches (time masking + jitter).",\n    )\n    parser.add_argument(\n        "--no-temporal-augmentation",\n        dest="temporal_augmentation",\n        action="store_false",\n        default=None,\n    )\n    parser.add_argument(\n        "--time-mask-ratio",\n        type=float,\n        default=None,\n        help="Fraction of observed timesteps masked in train augmentation (e.g. 0.05).",\n    )\n    parser.add_argument(\n        "--jitter-std",\n        type=float,\n        default=None,\n        help="Gaussian noise std applied on observed timesteps in train augmentation.",\n    )\n    parser.add_argument(\n        "--phase2-rare-finetune",\n        dest="phase2_rare_finetune",\n        action="store_true",\n        default=None,\n        help="Enable 2nd fine-tuning phase focused on rare classes.",\n    )\n    parser.add_argument(\n        "--no-phase2-rare-finetune",\n        dest="phase2_rare_finetune",\n        action="store_false",\n        default=None,\n    )\n    parser.add_argument("--phase2-epochs", type=int, default=None)\n    parser.add_argument("--phase2-lr", type=float, default=None)\n    parser.add_argument("--phase2-sampler-power", type=float, default=None)\n    parser.add_argument("--phase2-rare-quantile", type=float, default=None)\n    parser.add_argument("--phase2-rare-count-threshold", type=int, default=None)\n    parser.add_argument("--phase2-rare-boost", type=float, default=None)\n    parser.add_argument("--phase2-early-stopping-patience", type=int, default=None)\n    parser.add_argument("--early-stopping-patience", type=int, default=None)\n    parser.add_argument("--device", type=str, default=None, help="auto/cpu/cuda")\n    parser.add_argument("--seed", type=int, default=None)\n\n    return parser.parse_args()\n\n\ndef apply_args_to_config(args: argparse.Namespace, cfg: ExperimentConfig) -> ExperimentConfig:\n    if args.csv_path is not None:\n        cfg.data.csv_path = args.csv_path\n    if args.prepared_npz is not None:\n        cfg.data.prepared_npz_path = args.prepared_npz\n    if args.save_prepared_npz is not None:\n        cfg.data.save_prepared_npz_path = args.save_prepared_npz\n    if args.output_dir is not None:\n        cfg.data.output_dir = args.output_dir\n    if args.label_group_col is not None:\n        cfg.data.label_group_col = args.label_group_col\n\n    if args.split_method is not None:\n        cfg.data.split_method = args.split_method\n    if args.time_grid_frequency is not None:\n        cfg.data.time_grid_frequency = args.time_grid_frequency\n    if args.index_filter is not None:\n        cfg.data.index_filter = [x.strip() for x in args.index_filter.split(",") if x.strip()]\n    if args.min_obs is not None:\n        cfg.data.min_obs_per_parcel = args.min_obs\n    if args.max_cloud_scene is not None:\n        cfg.data.max_cloud_scene = args.max_cloud_scene\n    if args.min_px_count is not None:\n        cfg.data.min_px_count = args.min_px_count\n\n    if args.d_model is not None:\n        cfg.model.d_model = args.d_model\n    if args.n_heads is not None:\n        cfg.model.n_heads = args.n_heads\n    if args.n_layers is not None:\n        cfg.model.n_layers = args.n_layers\n    if args.ff_dim is not None:\n        cfg.model.dim_feedforward = args.ff_dim\n    if args.dropout is not None:\n        cfg.model.dropout = args.dropout\n    if args.pooling is not None:\n        cfg.model.pooling = args.pooling\n    if args.reliability_aware is not None:\n        cfg.model.reliability_aware = args.reliability_aware\n\n    if args.epochs is not None:\n        cfg.train.epochs = args.epochs\n    if args.batch_size is not None:\n        cfg.train.batch_size = args.batch_size\n    if args.lr is not None:\n        cfg.train.learning_rate = args.lr\n    if args.weight_decay is not None:\n        cfg.train.weight_decay = args.weight_decay\n    if args.standardize_features is not None:\n        cfg.train.standardize_features = args.standardize_features\n    if args.loss_type is not None:\n        cfg.train.loss_type = args.loss_type\n    if args.focal_gamma is not None:\n        cfg.train.focal_gamma = args.focal_gamma\n    if args.logit_adjust_tau is not None:\n        cfg.train.logit_adjust_tau = args.logit_adjust_tau\n    if args.use_group_task is not None:\n        cfg.train.use_group_task = args.use_group_task\n    if args.group_loss_weight is not None:\n        cfg.train.group_loss_weight = args.group_loss_weight\n    if args.hierarchical_constraint is not None:\n        cfg.train.hierarchical_constraint = args.hierarchical_constraint\n    if args.hierarchical_constraint_weight is not None:\n        cfg.train.hierarchical_constraint_weight = args.hierarchical_constraint_weight\n    if args.hierarchical_constraint_eps is not None:\n        cfg.train.hierarchical_constraint_eps = args.hierarchical_constraint_eps\n    if args.scheduler is not None:\n        cfg.train.scheduler = args.scheduler\n    if args.class_weight_power is not None:\n        cfg.train.class_weight_power = args.class_weight_power\n    if args.weighted_sampler is not None:\n        cfg.train.weighted_sampler = args.weighted_sampler\n    if args.sampler_power is not None:\n        cfg.train.sampler_power = args.sampler_power\n    if args.temporal_augmentation is not None:\n        cfg.train.temporal_augmentation = args.temporal_augmentation\n    if args.time_mask_ratio is not None:\n        cfg.train.time_mask_ratio = args.time_mask_ratio\n    if args.jitter_std is not None:\n        cfg.train.jitter_std = args.jitter_std\n    if args.phase2_rare_finetune is not None:\n        cfg.train.phase2_rare_finetune = args.phase2_rare_finetune\n    if args.phase2_epochs is not None:\n        cfg.train.phase2_epochs = args.phase2_epochs\n    if args.phase2_lr is not None:\n        cfg.train.phase2_learning_rate = args.phase2_lr\n    if args.phase2_sampler_power is not None:\n        cfg.train.phase2_sampler_power = args.phase2_sampler_power\n    if args.phase2_rare_quantile is not None:\n        cfg.train.phase2_rare_quantile = args.phase2_rare_quantile\n    if args.phase2_rare_count_threshold is not None:\n        cfg.train.phase2_rare_count_threshold = args.phase2_rare_count_threshold\n    if args.phase2_rare_boost is not None:\n        cfg.train.phase2_rare_boost = args.phase2_rare_boost\n    if args.phase2_early_stopping_patience is not None:\n        cfg.train.phase2_early_stopping_patience = args.phase2_early_stopping_patience\n    if args.early_stopping_patience is not None:\n        cfg.train.early_stopping_patience = args.early_stopping_patience\n    if args.device is not None:\n        cfg.train.device = args.device\n    if args.seed is not None:\n        cfg.train.seed = args.seed\n\n    if args.class_weighting:\n        cfg.train.class_weighting = True\n    if args.no_class_weighting:\n        cfg.train.class_weighting = False\n\n    return cfg\n\n\ndef _load_pretrained_encoder_weights(\n    model: nn.Module,\n    checkpoint_path: str,\n    device: torch.device,\n    logger,\n) -> None:\n    path = Path(checkpoint_path)\n    if not path.exists():\n        raise FileNotFoundError(f"pretrained encoder checkpoint not found: {path}")\n\n    try:\n        checkpoint = torch.load(path, map_location=device, weights_only=False)\n    except TypeError:\n        checkpoint = torch.load(path, map_location=device)\n\n    if isinstance(checkpoint, dict):\n        if "encoder_state_dict" in checkpoint and isinstance(checkpoint["encoder_state_dict"], dict):\n            source_state = checkpoint["encoder_state_dict"]\n        elif "model_state_dict" in checkpoint and isinstance(checkpoint["model_state_dict"], dict):\n            source_state = checkpoint["model_state_dict"]\n        else:\n            source_state = {\n                k: v\n                for k, v in checkpoint.items()\n                if isinstance(v, torch.Tensor)\n            }\n    else:\n        raise ValueError("Unsupported checkpoint format for pretrained encoder.")\n\n    allowed_prefixes = (\n        "feature_proj.",\n        "doy_encoding.",\n        "layers.",\n        "cls_token",\n        "reliability_proj.",\n        "reliability_gate_logit",\n    )\n    normalized_state: dict[str, torch.Tensor] = {}\n    for raw_key, value in source_state.items():\n        if not isinstance(value, torch.Tensor):\n            continue\n        key = str(raw_key)\n        if key.startswith("module."):\n            key = key[len("module.") :]\n        if key.startswith("backbone."):\n            key = key[len("backbone.") :]\n        normalized_state[key] = value\n\n    filtered_state = {\n        k: v\n        for k, v in normalized_state.items()\n        if any(k == prefix or k.startswith(prefix) for prefix in allowed_prefixes)\n    }\n    if len(filtered_state) == 0:\n        raise ValueError(\n            "No compatible encoder parameters found in pretrained checkpoint. "\n            "Expected keys starting with: feature_proj, doy_encoding, layers, cls_token, reliability_* "\n            "(optionally prefixed by backbone. and/or module.)."\n        )\n\n    missing, unexpected = model.load_state_dict(filtered_state, strict=False)\n    logger.info(\n        "Loaded pretrained encoder weights from %s | loaded=%d | missing=%d | unexpected=%d",\n        path,\n        len(filtered_state),\n        len(missing),\n        len(unexpected),\n    )\n\n\ndef build_scheduler(\n    optimizer: torch.optim.Optimizer,\n    cfg: ExperimentConfig,\n) -> Optional[torch.optim.lr_scheduler.LRScheduler]:\n    if cfg.train.scheduler == "none":\n        return None\n    if cfg.train.scheduler == "plateau":\n        return torch.optim.lr_scheduler.ReduceLROnPlateau(\n            optimizer,\n            mode="max",\n            factor=cfg.train.scheduler_factor,\n            patience=cfg.train.scheduler_patience,\n            min_lr=cfg.train.min_learning_rate,\n        )\n    if cfg.train.scheduler == "cosine":\n        return torch.optim.lr_scheduler.CosineAnnealingLR(\n            optimizer,\n            T_max=cfg.train.epochs,\n            eta_min=cfg.train.min_learning_rate,\n        )\n    raise ValueError(f"Unsupported scheduler: {cfg.train.scheduler}")\n\n\nclass FocalLoss(nn.Module):\n    def __init__(\n        self,\n        gamma: float = 2.0,\n        class_weights: Optional[torch.Tensor] = None,\n        ignore_index: Optional[int] = None,\n        reduction: str = "mean",\n    ) -> None:\n        super().__init__()\n        if gamma < 0:\n            raise ValueError("focal gamma must be >= 0.")\n        if reduction not in {"mean", "sum", "none"}:\n            raise ValueError("reduction must be one of: mean, sum, none.")\n        self.gamma = gamma\n        self.class_weights = class_weights\n        self.ignore_index = ignore_index\n        self.reduction = reduction\n\n    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:\n        if self.ignore_index is not None:\n            valid_mask = targets != self.ignore_index\n            if not torch.any(valid_mask):\n                return logits.sum() * 0.0\n            logits = logits[valid_mask]\n            targets = targets[valid_mask]\n\n        log_probs = F.log_softmax(logits, dim=1)\n        probs = log_probs.exp()\n\n        target_log_probs = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)\n        target_probs = probs.gather(1, targets.unsqueeze(1)).squeeze(1)\n\n        focal_factor = torch.pow((1.0 - target_probs).clamp(min=0.0), self.gamma)\n        loss = -focal_factor * target_log_probs\n\n        if self.class_weights is not None:\n            alpha = self.class_weights.gather(0, targets)\n            loss = alpha * loss\n\n        if self.reduction == "mean":\n            return loss.mean()\n        if self.reduction == "sum":\n            return loss.sum()\n        return loss\n\n\nclass BalancedSoftmaxLoss(nn.Module):\n    def __init__(\n        self,\n        class_counts: Tensor,\n        class_weights: Optional[Tensor] = None,\n        ignore_index: Optional[int] = None,\n        reduction: str = "mean",\n    ) -> None:\n        super().__init__()\n        if reduction not in {"mean", "sum", "none"}:\n            raise ValueError("reduction must be one of: mean, sum, none.")\n        counts = class_counts.float().clamp(min=1.0)\n        self.register_buffer("log_counts", counts.log())\n        self.class_weights = class_weights\n        self.ignore_index = ignore_index\n        self.reduction = reduction\n\n    def forward(self, logits: Tensor, targets: Tensor) -> Tensor:\n        if self.ignore_index is not None:\n            valid_mask = targets != self.ignore_index\n            if not torch.any(valid_mask):\n                return logits.sum() * 0.0\n            logits = logits[valid_mask]\n            targets = targets[valid_mask]\n\n        adjusted_logits = logits + self.log_counts.unsqueeze(0)\n        loss = F.cross_entropy(adjusted_logits, targets, reduction="none")\n        if self.class_weights is not None:\n            alpha = self.class_weights.gather(0, targets)\n            loss = alpha * loss\n\n        if self.reduction == "mean":\n            return loss.mean()\n        if self.reduction == "sum":\n            return loss.sum()\n        return loss\n\n\nclass LogitAdjustedCrossEntropyLoss(nn.Module):\n    def __init__(\n        self,\n        class_priors: Tensor,\n        tau: float = 1.0,\n        class_weights: Optional[Tensor] = None,\n        ignore_index: Optional[int] = None,\n        reduction: str = "mean",\n    ) -> None:\n        super().__init__()\n        if tau < 0:\n            raise ValueError("logit_adjust_tau must be >= 0.")\n        priors = class_priors.float().clamp(min=1e-12)\n        self.register_buffer("logit_adjustment", float(tau) * priors.log())\n        self.class_weights = class_weights\n        self.ignore_index = ignore_index\n        self.reduction = reduction\n\n    def forward(self, logits: Tensor, targets: Tensor) -> Tensor:\n        if self.ignore_index is not None:\n            valid_mask = targets != self.ignore_index\n            if not torch.any(valid_mask):\n                return logits.sum() * 0.0\n            logits = logits[valid_mask]\n            targets = targets[valid_mask]\n\n        adjusted_logits = logits + self.logit_adjustment.unsqueeze(0)\n        return F.cross_entropy(\n            adjusted_logits,\n            targets,\n            weight=self.class_weights,\n            reduction=self.reduction,\n        )\n\n\ndef build_loss_criterion(\n    loss_type: str,\n    class_weights: Optional[Tensor],\n    class_counts: Tensor,\n    class_priors: Tensor,\n    focal_gamma: float,\n    logit_adjust_tau: float,\n    ignore_index: Optional[int] = None,\n) -> nn.Module:\n    if loss_type == "cross_entropy":\n        return nn.CrossEntropyLoss(weight=class_weights, ignore_index=ignore_index if ignore_index is not None else -100)\n    if loss_type == "focal":\n        return FocalLoss(\n            gamma=focal_gamma,\n            class_weights=class_weights,\n            ignore_index=ignore_index,\n            reduction="mean",\n        )\n    if loss_type == "balanced_softmax":\n        return BalancedSoftmaxLoss(\n            class_counts=class_counts,\n            class_weights=class_weights,\n            ignore_index=ignore_index,\n            reduction="mean",\n        )\n    if loss_type == "logit_adjusted":\n        return LogitAdjustedCrossEntropyLoss(\n            class_priors=class_priors,\n            tau=logit_adjust_tau,\n            class_weights=class_weights,\n            ignore_index=ignore_index,\n            reduction="mean",\n        )\n    raise ValueError(f"Unsupported loss type: {loss_type}")\n\n\ndef build_weighted_train_sampler(\n    labels: np.ndarray,\n    train_indices: np.ndarray,\n    num_classes: int,\n    power: float,\n) -> tuple[WeightedRandomSampler, dict[str, float]]:\n    train_labels = labels[train_indices]\n    counts = np.bincount(train_labels, minlength=num_classes).astype(np.float64)\n    counts = np.maximum(counts, 1.0)\n    class_sample_weights = np.power(1.0 / counts, power)\n    class_sample_weights = class_sample_weights / np.clip(class_sample_weights.mean(), a_min=1e-12, a_max=None)\n    sample_weights = class_sample_weights[train_labels]\n    sampler = WeightedRandomSampler(\n        weights=torch.as_tensor(sample_weights, dtype=torch.double),\n        num_samples=len(train_labels),\n        replacement=True,\n    )\n    stats = {\n        "min_sample_weight": float(sample_weights.min()),\n        "max_sample_weight": float(sample_weights.max()),\n        "mean_sample_weight": float(sample_weights.mean()),\n    }\n    return sampler, stats\n\n\ndef select_rare_classes(\n    class_counts: np.ndarray,\n    rare_quantile: float,\n    rare_count_threshold: Optional[int],\n) -> tuple[np.ndarray, float]:\n    nonzero_counts = class_counts[class_counts > 0]\n    if nonzero_counts.size == 0:\n        return np.array([], dtype=np.int64), 0.0\n\n    if rare_count_threshold is not None:\n        threshold = float(rare_count_threshold)\n    else:\n        quantile = float(np.clip(rare_quantile, 0.0, 1.0))\n        threshold = float(np.quantile(nonzero_counts, quantile))\n\n    rare_ids = np.where(class_counts <= threshold)[0].astype(np.int64)\n    if rare_ids.size == 0:\n        rare_ids = np.array([int(np.argmin(class_counts))], dtype=np.int64)\n    return rare_ids, threshold\n\n\ndef build_rare_finetune_sampler(\n    labels: np.ndarray,\n    train_indices: np.ndarray,\n    num_classes: int,\n    rare_class_ids: np.ndarray,\n    power: float,\n    rare_boost: float,\n) -> tuple[WeightedRandomSampler, dict[str, float]]:\n    train_labels = labels[train_indices]\n    counts = np.bincount(train_labels, minlength=num_classes).astype(np.float64)\n    counts = np.maximum(counts, 1.0)\n\n    class_sample_weights = np.ones((num_classes,), dtype=np.float64)\n    if rare_class_ids.size > 0:\n        rare_scores = np.power(1.0 / counts[rare_class_ids], power)\n        rare_scores = rare_scores / np.clip(rare_scores.mean(), a_min=1e-12, a_max=None)\n        class_sample_weights[rare_class_ids] = float(rare_boost) * rare_scores\n\n    sample_weights = class_sample_weights[train_labels]\n    sampler = WeightedRandomSampler(\n        weights=torch.as_tensor(sample_weights, dtype=torch.double),\n        num_samples=len(train_labels),\n        replacement=True,\n    )\n    stats = {\n        "min_sample_weight": float(sample_weights.min()),\n        "max_sample_weight": float(sample_weights.max()),\n        "mean_sample_weight": float(sample_weights.mean()),\n    }\n    return sampler, stats\n\n\ndef build_class_group_compatibility(\n    labels: np.ndarray,\n    group_labels: np.ndarray,\n    train_indices: np.ndarray,\n    num_classes: int,\n    num_group_classes: int,\n) -> tuple[np.ndarray, dict[str, float]]:\n    if num_group_classes <= 0:\n        raise ValueError("num_group_classes must be > 0.")\n\n    train_labels = labels[train_indices]\n    train_groups = group_labels[train_indices]\n    valid_mask = train_groups >= 0\n    valid_labels = train_labels[valid_mask]\n    valid_groups = train_groups[valid_mask]\n\n    counts = np.zeros((num_group_classes, num_classes), dtype=np.float64)\n    np.add.at(counts, (valid_groups, valid_labels), 1.0)\n\n    col_sum = counts.sum(axis=0, keepdims=True)\n    missing_classes_mask = (col_sum <= 0).reshape(-1)\n    if np.any(missing_classes_mask):\n        counts[:, missing_classes_mask] = 1.0 / float(num_group_classes)\n        col_sum = counts.sum(axis=0, keepdims=True)\n\n    compat = counts / np.clip(col_sum, a_min=1.0e-12, a_max=None)\n    links_per_class = (compat > 0).sum(axis=0)\n    stats = {\n        "valid_samples": float(valid_mask.sum()),\n        "missing_classes": float(missing_classes_mask.sum()),\n        "mean_links_per_class": float(links_per_class.mean()),\n        "max_links_per_class": float(links_per_class.max()),\n    }\n    return compat.astype(np.float32), stats\n\n\ndef run_epoch(\n    model: nn.Module,\n    dataloader: torch.utils.data.DataLoader,\n    criterion: nn.Module,\n    device: torch.device,\n    optimizer: Optional[torch.optim.Optimizer] = None,\n    group_criterion: Optional[nn.Module] = None,\n    group_loss_weight: float = 0.0,\n    gradient_clip_norm: Optional[float] = None,\n) -> dict[str, float]:\n    is_train = optimizer is not None\n    model.train(mode=is_train)\n\n    total_loss = 0.0\n    total_group_loss = 0.0\n    all_targets: list[np.ndarray] = []\n    all_preds: list[np.ndarray] = []\n    total_samples = 0\n\n    progress = tqdm(dataloader, leave=False, desc="train" if is_train else "eval")\n    for batch in progress:\n        features = batch["features"].to(device=device, dtype=torch.float32)\n        day_of_year = batch["day_of_year"].to(device=device, dtype=torch.float32)\n        observed_mask = batch["observed_mask"].to(device=device, dtype=torch.bool)\n        quality_features = batch["quality_features"].to(device=device, dtype=torch.float32)\n        labels = batch["label"].to(device=device, dtype=torch.long)\n        group_labels = batch["group_label"].to(device=device, dtype=torch.long)\n\n        if is_train:\n            optimizer.zero_grad(set_to_none=True)\n\n        with torch.set_grad_enabled(is_train):\n            outputs = model(\n                features=features,\n                day_of_year=day_of_year,\n                observed_mask=observed_mask,\n                quality_features=quality_features,\n                return_attention=False,\n            )\n            logits = outputs["logits"]\n            main_loss = criterion(logits, labels)\n            group_loss = torch.zeros((), dtype=main_loss.dtype, device=main_loss.device)\n            if group_criterion is not None and "group_logits" in outputs:\n                group_loss = group_criterion(outputs["group_logits"], group_labels)\n            loss = main_loss + group_loss_weight * group_loss\n\n            if is_train:\n                loss.backward()\n                if gradient_clip_norm is not None:\n                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=gradient_clip_norm)\n                optimizer.step()\n\n        batch_size = labels.size(0)\n        total_loss += float(loss.item()) * batch_size\n        total_group_loss += float(group_loss.item()) * batch_size\n        total_samples += batch_size\n        all_targets.append(labels.detach().cpu().numpy())\n        all_preds.append(logits.argmax(dim=1).detach().cpu().numpy())\n\n    y_true = np.concatenate(all_targets) if all_targets else np.empty((0,), dtype=np.int64)\n    y_pred = np.concatenate(all_preds) if all_preds else np.empty((0,), dtype=np.int64)\n\n    epoch_loss = total_loss / max(total_samples, 1)\n    epoch_group_loss = total_group_loss / max(total_samples, 1)\n    epoch_acc = float(accuracy_score(y_true, y_pred)) if total_samples > 0 else float("nan")\n    epoch_recall_macro = (\n        float(recall_score(y_true, y_pred, average="macro", zero_division=0))\n        if total_samples > 0\n        else float("nan")\n    )\n    epoch_f1_weighted = (\n        float(f1_score(y_true, y_pred, average="weighted", zero_division=0))\n        if total_samples > 0\n        else float("nan")\n    )\n    epoch_f1 = float(f1_score(y_true, y_pred, average="macro", zero_division=0)) if total_samples > 0 else float("nan")\n    return {\n        "loss": epoch_loss,\n        "group_loss": epoch_group_loss,\n        "accuracy": epoch_acc,\n        "recall_macro": epoch_recall_macro,\n        "f1_weighted": epoch_f1_weighted,\n        "f1_macro": epoch_f1,\n    }\n\n\ndef save_eval_artifacts(\n    split_name: str,\n    metrics: dict,\n    label_names: list[str],\n    run_dir: Path,\n    time_grid: np.ndarray,\n) -> None:\n    serializable_metrics = {\n        "exactitude": metrics["accuracy"],\n        "precision_macro": metrics["precision_macro"],\n        "rappel_macro": metrics["recall_macro"],\n        "rappel_pondere": metrics["recall_weighted"],\n        "f1_macro": metrics["f1_macro"],\n        "f1_pondere": metrics["f1_weighted"],\n        "rapport_classification": metrics["classification_report_dict"],\n        "analyse_erreurs": metrics["error_analysis"],\n        "matrice_confusion": metrics["confusion_matrix"].tolist(),\n        # Compatibilite descendante\n        "accuracy": metrics["accuracy"],\n        "recall_macro": metrics["recall_macro"],\n        "recall_weighted": metrics["recall_weighted"],\n        "f1_weighted": metrics["f1_weighted"],\n        "classification_report": metrics["classification_report_dict"],\n        "error_analysis": metrics["error_analysis"],\n        "confusion_matrix": metrics["confusion_matrix"].tolist(),\n    }\n    save_json(serializable_metrics, run_dir / f"{split_name}_metriques.json")\n    save_json(serializable_metrics, run_dir / f"{split_name}_metrics.json")\n\n    with (run_dir / f"{split_name}_rapport_classification.txt").open("w", encoding="utf-8") as f:\n        f.write(metrics["classification_report_text"])\n    with (run_dir / f"{split_name}_classification_report.txt").open("w", encoding="utf-8") as f:\n        f.write(metrics["classification_report_text"])\n\n    plot_confusion_matrix(\n        confusion=metrics["confusion_matrix"],\n        label_names=label_names,\n        output_path=run_dir / f"{split_name}_matrice_confusion.png",\n        normalize=False,\n    )\n    plot_confusion_matrix(\n        confusion=metrics["confusion_matrix"],\n        label_names=label_names,\n        output_path=run_dir / f"{split_name}_matrice_confusion_normalisee.png",\n        normalize=True,\n    )\n    # Compatibilite descendante\n    plot_confusion_matrix(\n        confusion=metrics["confusion_matrix"],\n        label_names=label_names,\n        output_path=run_dir / f"{split_name}_confusion_matrix.png",\n        normalize=False,\n    )\n    plot_confusion_matrix(\n        confusion=metrics["confusion_matrix"],\n        label_names=label_names,\n        output_path=run_dir / f"{split_name}_confusion_matrix_normalized.png",\n        normalize=True,\n    )\n\n    predictions = metrics.get("predictions", {})\n    if "temporal_attention" in predictions:\n        attention_df = pd.DataFrame(\n            {\n                "date": pd.to_datetime(time_grid),\n                "day_of_year": pd.to_datetime(time_grid).dayofyear,\n                "mean_attention": predictions["temporal_attention"],\n            }\n        )\n        attention_df.to_csv(run_dir / f"{split_name}_attention_temporelle.csv", index=False)\n        attention_df.to_csv(run_dir / f"{split_name}_temporal_attention.csv", index=False)\n\n\ndef save_split_comparison_artifacts(split_metrics: dict[str, dict], run_dir: Path) -> None:\n    if not split_metrics:\n        return\n\n    split_order = list(split_metrics.keys())\n    split_labels = {\n        "train": "entrainement",\n        "val": "validation",\n        "validation": "validation",\n        "test": "test",\n    }\n\n    rows: list[dict[str, float | str]] = []\n    for split in split_order:\n        metrics = split_metrics[split]\n        rows.append(\n            {\n                "jeu": split_labels.get(split, split),\n                "exactitude": float(metrics.get("accuracy", float("nan"))),\n                "precision_macro": float(metrics.get("precision_macro", float("nan"))),\n                "rappel_macro": float(metrics.get("recall_macro", float("nan"))),\n                "f1_pondere": float(metrics.get("f1_weighted", float("nan"))),\n                "f1_macro": float(metrics.get("f1_macro", float("nan"))),\n            }\n        )\n\n    summary_df = pd.DataFrame(rows)\n    summary_df.to_csv(run_dir / "resume_metriques_splits.csv", index=False)\n    summary_df.to_csv(run_dir / "split_metrics_summary.csv", index=False)\n\ndef save_epoch_metric_curves(\n    history: list[dict[str, float]],\n    phase2_history: list[dict[str, float]],\n    run_dir: Path,\n) -> None:\n    if not history and not phase2_history:\n        return\n\n    all_rows: list[dict[str, float]] = []\n    for row in history:\n        all_rows.append(dict(row))\n\n    phase1_len = len(history)\n    for row in phase2_history:\n        merged = dict(row)\n        merged["epoch"] = float(phase1_len + int(row.get("epoch", 0)))\n        all_rows.append(merged)\n\n    history_df = pd.DataFrame(all_rows).sort_values("epoch").reset_index(drop=True)\n    history_df.to_csv(run_dir / "history_all_phases.csv", index=False)\n\n    metric_specs = [\n        ("accuracy", "Accuracy", "train_acc", "val_acc"),\n        ("recall_macro", "Recall (macro)", "train_recall_macro", "val_recall_macro"),\n        ("f1_weighted", "F1 (weighted)", "train_f1_weighted", "val_f1_weighted"),\n        ("f1_macro", "F1 (macro)", "train_f1_macro", "val_f1_macro"),\n    ]\n\n    for metric_key, metric_title, train_col, val_col in metric_specs:\n        if train_col not in history_df.columns or val_col not in history_df.columns:\n            continue\n\n        plt.figure(figsize=(8, 4.5))\n        plt.plot(\n            history_df["epoch"],\n            history_df[train_col],\n            marker="o",\n            linewidth=1.8,\n            markersize=4,\n            label="train",\n        )\n        plt.plot(\n            history_df["epoch"],\n            history_df[val_col],\n            marker="o",\n            linewidth=1.8,\n            markersize=4,\n            label="val",\n        )\n\n        if phase2_history:\n            plt.axvline(\n                x=phase1_len + 0.5,\n                color="gray",\n                linestyle="--",\n                linewidth=1.0,\n                label="phase2 start",\n            )\n\n        plt.ylim(0.0, 1.0)\n        plt.xlabel("Epoch")\n        plt.ylabel(metric_title)\n        plt.title(f"{metric_title}: train vs val")\n        plt.grid(alpha=0.25)\n        plt.legend()\n        plt.tight_layout()\n        plt.savefig(run_dir / f"metric_compare_{metric_key}.png", dpi=200)\n        plt.close()\n\n\ndef main() -> None:\n    args = parse_args()\n    cfg = apply_args_to_config(args, get_default_config())\n\n    if not 0.0 <= cfg.train.time_mask_ratio < 1.0:\n        raise ValueError("time_mask_ratio must be in [0, 1).")\n    if cfg.train.jitter_std < 0:\n        raise ValueError("jitter_std must be >= 0.")\n    if cfg.train.phase2_epochs <= 0:\n        raise ValueError("phase2_epochs must be > 0.")\n    if cfg.train.phase2_learning_rate <= 0:\n        raise ValueError("phase2_learning_rate must be > 0.")\n    if cfg.train.phase2_sampler_power < 0:\n        raise ValueError("phase2_sampler_power must be >= 0.")\n    if not 0.0 <= cfg.train.phase2_rare_quantile <= 1.0:\n        raise ValueError("phase2_rare_quantile must be in [0, 1].")\n    if cfg.train.phase2_rare_boost <= 0:\n        raise ValueError("phase2_rare_boost must be > 0.")\n    if cfg.train.phase2_early_stopping_patience <= 0:\n        raise ValueError("phase2_early_stopping_patience must be > 0.")\n    if cfg.train.hierarchical_constraint_weight < 0:\n        raise ValueError("hierarchical_constraint_weight must be >= 0.")\n    if cfg.train.hierarchical_constraint_eps <= 0:\n        raise ValueError("hierarchical_constraint_eps must be > 0.")\n\n    set_seed(cfg.train.seed)\n    run_dir = create_run_dir(cfg.data.output_dir, prefix="temporal_transformer")\n    logger = setup_logger(run_dir / "train.log")\n    logger.info("Run directory: %s", run_dir)\n    logger.info("Configuration loaded.")\n    save_json(cfg.to_dict(), run_dir / "config.json")\n\n    prepared = prepare_dataset(cfg.data)\n    logger.info(\n        "Prepared dataset: N=%d, T=%d, F=%d, classes=%d",\n        prepared.features.shape[0],\n        prepared.seq_len,\n        prepared.num_features,\n        prepared.num_classes,\n    )\n    logger.info("Features: %s", ", ".join(prepared.feature_names))\n    logger.info("Classes: %s", ", ".join(prepared.label_names))\n    if prepared.has_group_labels:\n        logger.info("Group classes (%s): %s", cfg.data.label_group_col, ", ".join(prepared.group_label_names))\n    else:\n        logger.info("No usable group labels detected in column \'%s\'.", cfg.data.label_group_col)\n    split_sizes = {k: int(v.shape[0]) for k, v in prepared.splits.items()}\n    logger.info("Split sizes: %s", split_sizes)\n\n    if split_sizes.get("train", 0) == 0:\n        raise ValueError("Le split d\'entrainement est vide.")\n    if split_sizes.get("val", 0) == 0:\n        logger.warning("Le split de validation est vide. L\'early stopping sera desactive.")\n    if split_sizes.get("test", 0) == 0:\n        raise ValueError("Le split de test est vide.")\n\n    use_group_task = cfg.train.use_group_task or cfg.train.hierarchical_constraint\n    if cfg.train.hierarchical_constraint and not cfg.train.use_group_task:\n        logger.info(\n            "Hierarchical constrained decoding enabled: group head is activated automatically."\n        )\n    if use_group_task and not prepared.has_group_labels:\n        raise ValueError(\n            "Group task enabled but no group labels are available. "\n            f"Ensure column \'{cfg.data.label_group_col}\' exists and is populated."\n        )\n    if use_group_task and cfg.train.group_loss_weight <= 0:\n        raise ValueError("group_loss_weight must be > 0 when group task is enabled.")\n\n    if cfg.train.standardize_features:\n        scaler_stats = standardize_prepared_features(\n            prepared=prepared,\n            train_indices=prepared.splits["train"],\n            eps=cfg.train.standardize_eps,\n        )\n        save_json(\n            {\n                "feature_names": prepared.feature_names,\n                "mean": scaler_stats["mean"].tolist(),\n                "std": scaler_stats["std"].tolist(),\n                "count_per_feature": scaler_stats["count_per_feature"].tolist(),\n            },\n            run_dir / "feature_scaler.json",\n        )\n        logger.info(\n            "Applied train-only feature standardization (eps=%.1e).",\n            cfg.train.standardize_eps,\n        )\n    else:\n        logger.info("Feature standardization disabled.")\n\n    device = resolve_device(cfg.train.device)\n    logger.info("Device: %s", device)\n\n    train_augmentation: Optional[TemporalAugmentationConfig] = None\n    if cfg.train.temporal_augmentation:\n        train_augmentation = TemporalAugmentationConfig(\n            time_mask_ratio=cfg.train.time_mask_ratio,\n            jitter_std=cfg.train.jitter_std,\n        )\n        logger.info(\n            "Temporal augmentation enabled | time_mask_ratio=%.3f | jitter_std=%.4f",\n            cfg.train.time_mask_ratio,\n            cfg.train.jitter_std,\n        )\n    else:\n        logger.info("Temporal augmentation disabled.")\n\n    train_sampler: Optional[WeightedRandomSampler] = None\n    if cfg.train.weighted_sampler:\n        train_sampler, sampler_stats = build_weighted_train_sampler(\n            labels=prepared.labels,\n            train_indices=prepared.splits["train"],\n            num_classes=prepared.num_classes,\n            power=cfg.train.sampler_power,\n        )\n        logger.info(\n            "Weighted sampler enabled | power=%.3f | min=%.3f max=%.3f",\n            cfg.train.sampler_power,\n            sampler_stats["min_sample_weight"],\n            sampler_stats["max_sample_weight"],\n        )\n    else:\n        logger.info("Weighted sampler disabled.")\n\n    if cfg.train.weighted_sampler and cfg.train.class_weighting:\n        logger.warning(\n            "Both weighted sampler and class weighting are enabled. "\n            "This can over-correct minority classes; monitor precision/recall tradeoff."\n        )\n    if cfg.train.phase2_rare_finetune and cfg.train.weighted_sampler:\n        logger.warning(\n            "phase2_rare_finetune is enabled, but weighted sampler is also active in phase 1. "\n            "For a stricter two-stage setup, use --no-weighted-sampler in phase 1."\n        )\n\n    dataloaders = build_dataloaders(\n        prepared=prepared,\n        batch_size=cfg.train.batch_size,\n        num_workers=cfg.train.num_workers,\n        pin_memory=(device.type == "cuda"),\n        train_sampler=train_sampler,\n        train_augmentation=train_augmentation,\n    )\n\n    model = TemporalTransformerClassifier(\n        input_dim=prepared.num_features,\n        num_classes=prepared.num_classes,\n        cfg=cfg.model,\n        num_group_classes=prepared.num_group_classes if use_group_task else 0,\n    ).to(device)\n    logger.info("Reliability-aware transformer: %s", "enabled" if cfg.model.reliability_aware else "disabled")\n    if args.pretrained_encoder_checkpoint:\n        _load_pretrained_encoder_weights(\n            model=model,\n            checkpoint_path=args.pretrained_encoder_checkpoint,\n            device=device,\n            logger=logger,\n        )\n\n    train_labels = prepared.labels[prepared.splits["train"]]\n    class_counts_np = np.bincount(train_labels, minlength=prepared.num_classes).astype(np.float64)\n    class_counts_np = np.clip(class_counts_np, a_min=1.0, a_max=None)\n    class_priors_np = class_counts_np / class_counts_np.sum()\n    class_counts_t = torch.tensor(class_counts_np, dtype=torch.float32, device=device)\n    class_priors_t = torch.tensor(class_priors_np, dtype=torch.float32, device=device)\n\n    class_weights: Optional[torch.Tensor] = None\n    if cfg.train.class_weighting:\n        class_weights_np = compute_class_weights(\n            train_labels,\n            num_classes=prepared.num_classes,\n            power=cfg.train.class_weight_power,\n        )\n        class_weights = torch.tensor(class_weights_np, dtype=torch.float32, device=device)\n        logger.info(\n            "Class weighting enabled | power=%.3f | min=%.3f max=%.3f",\n            cfg.train.class_weight_power,\n            float(class_weights.min().item()),\n            float(class_weights.max().item()),\n        )\n    else:\n        logger.info("Class weighting disabled.")\n\n    if cfg.train.loss_type == "balanced_softmax" and cfg.train.class_weighting:\n        logger.warning(\n            "balanced_softmax with class weighting can over-correct tail classes; "\n            "consider disabling class weighting for this loss."\n        )\n\n    group_class_weights: Optional[torch.Tensor] = None\n    group_criterion: Optional[nn.Module] = None\n    group_counts_t: Optional[torch.Tensor] = None\n    group_priors_t: Optional[torch.Tensor] = None\n    class_group_compat_np: Optional[np.ndarray] = None\n    if use_group_task:\n        assert prepared.group_labels is not None\n        train_group_labels = prepared.group_labels[prepared.splits["train"]]\n        valid_group_train = train_group_labels[train_group_labels >= 0]\n        if valid_group_train.size == 0:\n            raise ValueError("Group task enabled but no valid group labels found in train split.")\n\n        group_counts_np = np.bincount(valid_group_train, minlength=prepared.num_group_classes).astype(np.float64)\n        group_counts_np = np.clip(group_counts_np, a_min=1.0, a_max=None)\n        group_priors_np = group_counts_np / group_counts_np.sum()\n        group_counts_t = torch.tensor(group_counts_np, dtype=torch.float32, device=device)\n        group_priors_t = torch.tensor(group_priors_np, dtype=torch.float32, device=device)\n\n        if cfg.train.class_weighting:\n            group_class_weights_np = compute_class_weights(\n                valid_group_train,\n                num_classes=prepared.num_group_classes,\n                power=cfg.train.class_weight_power,\n            )\n            group_class_weights = torch.tensor(group_class_weights_np, dtype=torch.float32, device=device)\n            logger.info(\n                "Group class weighting enabled | power=%.3f | min=%.3f max=%.3f",\n                cfg.train.class_weight_power,\n                float(group_class_weights.min().item()),\n                float(group_class_weights.max().item()),\n            )\n\n        logger.info(\n            "Group task enabled | groups=%d | group_loss_weight=%.3f",\n            prepared.num_group_classes,\n            cfg.train.group_loss_weight,\n        )\n\n        if cfg.train.hierarchical_constraint:\n            class_group_compat_np, compat_stats = build_class_group_compatibility(\n                labels=prepared.labels,\n                group_labels=prepared.group_labels,\n                train_indices=prepared.splits["train"],\n                num_classes=prepared.num_classes,\n                num_group_classes=prepared.num_group_classes,\n            )\n            model.configure_hierarchical_constraint(\n                class_group_compat=torch.tensor(class_group_compat_np, dtype=torch.float32, device=device),\n                weight=cfg.train.hierarchical_constraint_weight,\n                eps=cfg.train.hierarchical_constraint_eps,\n                enabled=True,\n            )\n            save_json(\n                {\n                    "group_label_names": prepared.group_label_names,\n                    "label_names": prepared.label_names,\n                    "class_group_compat": class_group_compat_np.tolist(),\n                    "weight": cfg.train.hierarchical_constraint_weight,\n                    "eps": cfg.train.hierarchical_constraint_eps,\n                    "stats": compat_stats,\n                },\n                run_dir / "hierarchical_constraint.json",\n            )\n            logger.info(\n                "Hierarchical constrained decoding enabled | weight=%.3f | eps=%.1e | missing_classes=%.0f | mean_links=%.2f",\n                cfg.train.hierarchical_constraint_weight,\n                cfg.train.hierarchical_constraint_eps,\n                compat_stats["missing_classes"],\n                compat_stats["mean_links_per_class"],\n            )\n\n    criterion = build_loss_criterion(\n        loss_type=cfg.train.loss_type,\n        class_weights=class_weights,\n        class_counts=class_counts_t,\n        class_priors=class_priors_t,\n        focal_gamma=cfg.train.focal_gamma,\n        logit_adjust_tau=cfg.train.logit_adjust_tau,\n        ignore_index=None,\n    )\n    if use_group_task:\n        assert group_counts_t is not None and group_priors_t is not None\n        group_criterion = build_loss_criterion(\n            loss_type=cfg.train.loss_type,\n            class_weights=group_class_weights,\n            class_counts=group_counts_t,\n            class_priors=group_priors_t,\n            focal_gamma=cfg.train.focal_gamma,\n            logit_adjust_tau=cfg.train.logit_adjust_tau,\n            ignore_index=-100,\n        )\n\n    if cfg.train.loss_type == "cross_entropy":\n        logger.info("Loss: CrossEntropy")\n    elif cfg.train.loss_type == "focal":\n        logger.info("Loss: Focal (gamma=%.3f)", cfg.train.focal_gamma)\n    elif cfg.train.loss_type == "balanced_softmax":\n        logger.info("Loss: BalancedSoftmax")\n    elif cfg.train.loss_type == "logit_adjusted":\n        logger.info("Loss: LogitAdjustedCrossEntropy (tau=%.3f)", cfg.train.logit_adjust_tau)\n\n    optimizer = torch.optim.AdamW(\n        model.parameters(),\n        lr=cfg.train.learning_rate,\n        weight_decay=cfg.train.weight_decay,\n    )\n    scheduler = build_scheduler(optimizer, cfg)\n    early_stopping = (\n        EarlyStopping(patience=cfg.train.early_stopping_patience, mode="max")\n        if split_sizes.get("val", 0) > 0\n        else None\n    )\n\n    best_val_f1 = float("-inf")\n    best_model_path = run_dir / "best_model.pt"\n    history: list[dict[str, float]] = []\n    phase2_history: list[dict[str, float]] = []\n\n    for epoch in range(1, cfg.train.epochs + 1):\n        train_metrics = run_epoch(\n            model=model,\n            dataloader=dataloaders["train"],\n            criterion=criterion,\n            device=device,\n            optimizer=optimizer,\n            group_criterion=group_criterion,\n            group_loss_weight=cfg.train.group_loss_weight if use_group_task else 0.0,\n            gradient_clip_norm=cfg.train.gradient_clip_norm,\n        )\n\n        if split_sizes.get("val", 0) > 0:\n            val_metrics = run_epoch(\n                model=model,\n                dataloader=dataloaders["val"],\n                criterion=criterion,\n                device=device,\n                optimizer=None,\n                group_criterion=group_criterion,\n                group_loss_weight=cfg.train.group_loss_weight if use_group_task else 0.0,\n                gradient_clip_norm=None,\n            )\n            val_f1 = val_metrics["f1_macro"]\n        else:\n            val_metrics = {\n                "loss": float("nan"),\n                "group_loss": float("nan"),\n                "accuracy": float("nan"),\n                "recall_macro": float("nan"),\n                "f1_weighted": float("nan"),\n                "f1_macro": float("nan"),\n            }\n            val_f1 = train_metrics["f1_macro"]\n\n        current_lr = float(optimizer.param_groups[0]["lr"])\n        if use_group_task:\n            logger.info(\n                "Epoch %03d | lr=%.6f | train_loss=%.4f (grp=%.4f) train_f1=%.4f | "\n                "val_loss=%.4f (grp=%.4f) val_f1=%.4f",\n                epoch,\n                current_lr,\n                train_metrics["loss"],\n                train_metrics["group_loss"],\n                train_metrics["f1_macro"],\n                val_metrics["loss"],\n                val_metrics["group_loss"],\n                val_metrics["f1_macro"],\n            )\n        else:\n            logger.info(\n                "Epoch %03d | lr=%.6f | train_loss=%.4f train_f1=%.4f | val_loss=%.4f val_f1=%.4f",\n                epoch,\n                current_lr,\n                train_metrics["loss"],\n                train_metrics["f1_macro"],\n                val_metrics["loss"],\n                val_metrics["f1_macro"],\n            )\n        history.append(\n            {\n                "epoch": float(epoch),\n                "lr": current_lr,\n                "train_loss": train_metrics["loss"],\n                "train_group_loss": train_metrics["group_loss"],\n                "train_acc": train_metrics["accuracy"],\n                "train_recall_macro": train_metrics["recall_macro"],\n                "train_f1_weighted": train_metrics["f1_weighted"],\n                "train_f1_macro": train_metrics["f1_macro"],\n                "val_loss": val_metrics["loss"],\n                "val_group_loss": val_metrics["group_loss"],\n                "val_acc": val_metrics["accuracy"],\n                "val_recall_macro": val_metrics["recall_macro"],\n                "val_f1_weighted": val_metrics["f1_weighted"],\n                "val_f1_macro": val_metrics["f1_macro"],\n            }\n        )\n\n        improved = val_f1 > best_val_f1\n        if improved:\n            best_val_f1 = val_f1\n            save_checkpoint(\n                path=best_model_path,\n                model=model,\n                optimizer=optimizer,\n                scheduler=scheduler,\n                epoch=epoch,\n                best_metric=best_val_f1,\n                metadata={\n                    "label_names": prepared.label_names,\n                    "group_label_names": prepared.group_label_names if use_group_task else [],\n                    "feature_names": prepared.feature_names,\n                    "pooling": cfg.model.pooling,\n                    "reliability_aware": bool(cfg.model.reliability_aware),\n                    "loss_type": cfg.train.loss_type,\n                    "focal_gamma": cfg.train.focal_gamma,\n                    "logit_adjust_tau": cfg.train.logit_adjust_tau,\n                    "class_weight_power": cfg.train.class_weight_power,\n                    "num_group_classes": prepared.num_group_classes if use_group_task else 0,\n                    "use_group_task": bool(use_group_task),\n                    "hierarchical_constraint": bool(cfg.train.hierarchical_constraint),\n                    "hierarchical_constraint_weight": cfg.train.hierarchical_constraint_weight,\n                    "hierarchical_constraint_eps": cfg.train.hierarchical_constraint_eps,\n                    "class_group_compat": class_group_compat_np.tolist() if class_group_compat_np is not None else None,\n                },\n            )\n            logger.info("Saved new best model (val_f1=%.4f).", best_val_f1)\n\n        if scheduler is not None:\n            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):\n                scheduler.step(val_f1)\n            else:\n                scheduler.step()\n\n        if early_stopping is not None and early_stopping.step(val_f1):\n            logger.info("Early stopping triggered at epoch %d.", epoch)\n            break\n\n    pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)\n\n    if cfg.train.phase2_rare_finetune:\n        logger.info("Starting phase 2 rare-class fine-tuning.")\n        try:\n            checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)\n        except TypeError:\n            checkpoint = torch.load(best_model_path, map_location=device)\n        model.load_state_dict(checkpoint["model_state_dict"])\n\n        rare_class_ids, rare_threshold = select_rare_classes(\n            class_counts=class_counts_np,\n            rare_quantile=cfg.train.phase2_rare_quantile,\n            rare_count_threshold=cfg.train.phase2_rare_count_threshold,\n        )\n        rare_class_names = [prepared.label_names[int(i)] for i in rare_class_ids.tolist()]\n        logger.info(\n            "Phase 2 rare classes selected | threshold=%.1f | n_rare=%d | classes=%s",\n            rare_threshold,\n            len(rare_class_ids),\n            ", ".join(rare_class_names),\n        )\n\n        phase2_sampler, phase2_stats = build_rare_finetune_sampler(\n            labels=prepared.labels,\n            train_indices=prepared.splits["train"],\n            num_classes=prepared.num_classes,\n            rare_class_ids=rare_class_ids,\n            power=cfg.train.phase2_sampler_power,\n            rare_boost=cfg.train.phase2_rare_boost,\n        )\n        logger.info(\n            "Phase 2 sampler enabled | power=%.3f | boost=%.3f | min=%.3f max=%.3f",\n            cfg.train.phase2_sampler_power,\n            cfg.train.phase2_rare_boost,\n            phase2_stats["min_sample_weight"],\n            phase2_stats["max_sample_weight"],\n        )\n\n        phase2_dataloaders = build_dataloaders(\n            prepared=prepared,\n            batch_size=cfg.train.batch_size,\n            num_workers=cfg.train.num_workers,\n            pin_memory=(device.type == "cuda"),\n            train_sampler=phase2_sampler,\n            train_augmentation=train_augmentation,\n        )\n\n        phase2_optimizer = torch.optim.AdamW(\n            model.parameters(),\n            lr=cfg.train.phase2_learning_rate,\n            weight_decay=cfg.train.weight_decay,\n        )\n        phase2_scheduler = None\n        phase2_early_stopping = (\n            EarlyStopping(patience=cfg.train.phase2_early_stopping_patience, mode="max")\n            if split_sizes.get("val", 0) > 0\n            else None\n        )\n\n        phase2_best_model_path = run_dir / "best_model_phase2.pt"\n        phase2_best_val_f1 = best_val_f1\n        for epoch in range(1, cfg.train.phase2_epochs + 1):\n            train_metrics = run_epoch(\n                model=model,\n                dataloader=phase2_dataloaders["train"],\n                criterion=criterion,\n                device=device,\n                optimizer=phase2_optimizer,\n                group_criterion=group_criterion,\n                group_loss_weight=cfg.train.group_loss_weight if use_group_task else 0.0,\n                gradient_clip_norm=cfg.train.gradient_clip_norm,\n            )\n\n            if split_sizes.get("val", 0) > 0:\n                val_metrics = run_epoch(\n                    model=model,\n                    dataloader=phase2_dataloaders["val"],\n                    criterion=criterion,\n                    device=device,\n                    optimizer=None,\n                    group_criterion=group_criterion,\n                    group_loss_weight=cfg.train.group_loss_weight if use_group_task else 0.0,\n                    gradient_clip_norm=None,\n                )\n                val_f1 = val_metrics["f1_macro"]\n            else:\n                val_metrics = {\n                    "loss": float("nan"),\n                    "group_loss": float("nan"),\n                    "accuracy": float("nan"),\n                    "recall_macro": float("nan"),\n                    "f1_weighted": float("nan"),\n                    "f1_macro": float("nan"),\n                }\n                val_f1 = train_metrics["f1_macro"]\n\n            current_lr = float(phase2_optimizer.param_groups[0]["lr"])\n            if use_group_task:\n                logger.info(\n                    "Phase2 Epoch %03d | lr=%.6f | train_loss=%.4f (grp=%.4f) train_f1=%.4f | "\n                    "val_loss=%.4f (grp=%.4f) val_f1=%.4f",\n                    epoch,\n                    current_lr,\n                    train_metrics["loss"],\n                    train_metrics["group_loss"],\n                    train_metrics["f1_macro"],\n                    val_metrics["loss"],\n                    val_metrics["group_loss"],\n                    val_metrics["f1_macro"],\n                )\n            else:\n                logger.info(\n                    "Phase2 Epoch %03d | lr=%.6f | train_loss=%.4f train_f1=%.4f | val_loss=%.4f val_f1=%.4f",\n                    epoch,\n                    current_lr,\n                    train_metrics["loss"],\n                    train_metrics["f1_macro"],\n                    val_metrics["loss"],\n                    val_metrics["f1_macro"],\n                )\n\n            phase2_history.append(\n                {\n                    "epoch": float(epoch),\n                    "lr": current_lr,\n                    "train_loss": train_metrics["loss"],\n                    "train_group_loss": train_metrics["group_loss"],\n                    "train_acc": train_metrics["accuracy"],\n                    "train_recall_macro": train_metrics["recall_macro"],\n                    "train_f1_weighted": train_metrics["f1_weighted"],\n                    "train_f1_macro": train_metrics["f1_macro"],\n                    "val_loss": val_metrics["loss"],\n                    "val_group_loss": val_metrics["group_loss"],\n                    "val_acc": val_metrics["accuracy"],\n                    "val_recall_macro": val_metrics["recall_macro"],\n                    "val_f1_weighted": val_metrics["f1_weighted"],\n                    "val_f1_macro": val_metrics["f1_macro"],\n                }\n            )\n\n            if val_f1 > phase2_best_val_f1:\n                phase2_best_val_f1 = val_f1\n                save_checkpoint(\n                    path=phase2_best_model_path,\n                    model=model,\n                    optimizer=phase2_optimizer,\n                    scheduler=phase2_scheduler,\n                    epoch=epoch,\n                    best_metric=phase2_best_val_f1,\n                    metadata={\n                        "label_names": prepared.label_names,\n                        "group_label_names": prepared.group_label_names if use_group_task else [],\n                        "feature_names": prepared.feature_names,\n                        "pooling": cfg.model.pooling,\n                        "reliability_aware": bool(cfg.model.reliability_aware),\n                        "loss_type": cfg.train.loss_type,\n                        "focal_gamma": cfg.train.focal_gamma,\n                        "logit_adjust_tau": cfg.train.logit_adjust_tau,\n                        "class_weight_power": cfg.train.class_weight_power,\n                        "num_group_classes": prepared.num_group_classes if use_group_task else 0,\n                        "use_group_task": bool(use_group_task),\n                        "stage": "phase2_rare_finetune",\n                        "phase2_learning_rate": cfg.train.phase2_learning_rate,\n                        "rare_class_ids": rare_class_ids.tolist(),\n                        "hierarchical_constraint": bool(cfg.train.hierarchical_constraint),\n                        "hierarchical_constraint_weight": cfg.train.hierarchical_constraint_weight,\n                        "hierarchical_constraint_eps": cfg.train.hierarchical_constraint_eps,\n                        "class_group_compat": class_group_compat_np.tolist() if class_group_compat_np is not None else None,\n                    },\n                )\n                logger.info("Saved new phase 2 best model (val_f1=%.4f).", phase2_best_val_f1)\n\n            if phase2_early_stopping is not None and phase2_early_stopping.step(val_f1):\n                logger.info("Phase 2 early stopping triggered at epoch %d.", epoch)\n                break\n\n        pd.DataFrame(phase2_history).to_csv(run_dir / "history_phase2.csv", index=False)\n\n        if phase2_best_model_path.exists() and phase2_best_val_f1 > best_val_f1:\n            best_val_f1 = phase2_best_val_f1\n            best_model_path.write_bytes(phase2_best_model_path.read_bytes())\n            logger.info(\n                "Phase 2 improved best val_f1 to %.4f. Promoted %s -> %s",\n                best_val_f1,\n                phase2_best_model_path.name,\n                best_model_path.name,\n            )\n        else:\n            logger.info(\n                "Phase 2 did not improve over phase 1 best val_f1=%.4f. Keeping phase 1 checkpoint.",\n                best_val_f1,\n            )\n\n    save_epoch_metric_curves(history=history, phase2_history=phase2_history, run_dir=run_dir)\n\n    logger.info("Training finished. Loading best checkpoint from %s", best_model_path)\n    try:\n        checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)\n    except TypeError:\n        checkpoint = torch.load(best_model_path, map_location=device)\n    model.load_state_dict(checkpoint["model_state_dict"])\n\n    eval_dataloaders = build_dataloaders(\n        prepared=prepared,\n        batch_size=cfg.train.batch_size,\n        num_workers=cfg.train.num_workers,\n        pin_memory=(device.type == "cuda"),\n        train_sampler=None,\n        train_augmentation=None,\n    )\n    split_evals: dict[str, dict] = {}\n\n    test_eval = evaluate_split(\n        model=model,\n        dataloader=eval_dataloaders["test"],\n        device=device,\n        label_names=prepared.label_names,\n        pooling=cfg.model.pooling,\n        return_attention=True,\n    )\n    save_eval_artifacts("test", test_eval, prepared.label_names, run_dir, prepared.time_grid)\n    split_evals["test"] = test_eval\n    logger.info(\n        "TEST | exactitude=%.4f precision_macro=%.4f rappel_macro=%.4f f1_macro=%.4f f1_pondere=%.4f",\n        test_eval["accuracy"],\n        test_eval["precision_macro"],\n        test_eval["recall_macro"],\n        test_eval["f1_macro"],\n        test_eval["f1_weighted"],\n    )\n\n    save_split_comparison_artifacts(split_evals, run_dir)\n    logger.info("Artefacts de test sauvegardes dans %s", run_dir)\n\n\nif __name__ == "__main__":\n    main()\n'

def load_embedded_module(name: str):
    mod = types.ModuleType(name)
    mod.__file__ = f'<embedded:{name}>'
    sys.modules[name] = mod
    exec(embedded_sources[name], mod.__dict__)
    return mod

loaded_modules = {}
module_order = ['config', 'utils', 'data', 'model', 'evaluate', 'evaluate_ensemble', 'pretrain_ssl', 'train']
for _name in module_order:
    loaded_modules[_name] = load_embedded_module(_name)

print('Modules charges:', ', '.join(module_order))


## Cellule 4 - Helpers d'exécution des variantes

Expose des fonctions pour lancer l'entraînement, le SSL et l'ensemble depuis le code embarqué.


In [ ]:
import sys
import copy
import json
from pathlib import Path

config_mod = loaded_modules["config"]
train_mod = loaded_modules["train"]
ssl_mod = loaded_modules["pretrain_ssl"]
ens_mod = loaded_modules["evaluate_ensemble"]


def run_module_main(module, argv: list[str]) -> None:
    old_argv = sys.argv[:]
    try:
        sys.argv = [module.__name__ + ".py"] + [str(x) for x in argv]
        module.main()
    finally:
        sys.argv = old_argv


def run_train_variant(argv: list[str]) -> None:
    run_module_main(train_mod, argv)


def run_ssl_pretrain(argv: list[str]) -> None:
    run_module_main(ssl_mod, argv)


def run_ensemble_eval(argv: list[str]) -> None:
    run_module_main(ens_mod, argv)


def latest_run_dir(base_output: Path) -> Path | None:
    candidates = sorted(
        [p for p in base_output.rglob("temporal_transformer_*") if p.is_dir()],
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    return candidates[0] if candidates else None


def read_test_metrics(run_dir: Path) -> dict:
    path = run_dir / "test_metrics.json"
    if not path.exists():
        path = run_dir / "test_metriques.json"
    if not path.exists():
        raise FileNotFoundError(f"Metrics introuvables dans {run_dir}")
    return json.loads(path.read_text(encoding="utf-8"))


## Cellule 5 - Définition des variantes

Toutes les variantes sont listées ici. La **variante finale** est explicitement marquée.


In [ ]:
INDEX_FILTER = "NDVI,NDMI,NDWI,GNDVI,SAVI,OSAVI,MSAVI,MNDWI,ARVI,BSI"

# Fournir ce chemin pour la variante finale (SSL -> supervise)
SSL_ENCODER_PATH = ""  # ex: outputs_transformer/ssl_pretrain_ext/ssl_pretrain_YYYYMMDD_HHMMSS/best_ssl_encoder.pt

VARIANTES = {
    "v1_baseline": {
        "description": "Baseline: CrossEntropy sans ponderation.",
        "is_final": False,
        "args": [
            "--prepared-npz", str(PREPARED_NPZ),
            "--split-method", "tile",
            "--index-filter", INDEX_FILTER,
            "--loss-type", "cross_entropy",
            "--no-class-weighting",
            "--no-weighted-sampler",
            "--output-dir", str(OUTPUT_ROOT / "v1_baseline"),
        ],
    },
    "v2_focal": {
        "description": "Focal + class weights + standardisation.",
        "is_final": False,
        "args": [
            "--prepared-npz", str(PREPARED_NPZ),
            "--split-method", "tile",
            "--index-filter", INDEX_FILTER,
            "--loss-type", "focal",
            "--focal-gamma", "1.5",
            "--class-weighting",
            "--class-weight-power", "0.5",
            "--standardize-features",
            "--no-weighted-sampler",
            "--output-dir", str(OUTPUT_ROOT / "v2_focal"),
        ],
    },
    "v3_phase2_rares": {
        "description": "Focal + phase 2 classes rares.",
        "is_final": False,
        "args": [
            "--prepared-npz", str(PREPARED_NPZ),
            "--split-method", "tile",
            "--index-filter", INDEX_FILTER,
            "--loss-type", "focal",
            "--focal-gamma", "1.5",
            "--class-weighting",
            "--class-weight-power", "0.5",
            "--standardize-features",
            "--no-weighted-sampler",
            "--phase2-rare-finetune",
            "--phase2-epochs", "12",
            "--phase2-lr", "1e-4",
            "--phase2-sampler-power", "1.0",
            "--phase2-rare-quantile", "0.25",
            "--phase2-rare-boost", "2.0",
            "--output-dir", str(OUTPUT_ROOT / "v3_phase2_rares"),
        ],
    },
    "v4_ssl_reliability_finale": {
        "description": "VARIANTE FINALE: reliability-aware + init SSL + phase 2 classes rares.",
        "is_final": True,
        "requires_ssl": True,
        "args": [
            "--prepared-npz", str(PREPARED_NPZ),
            "--split-method", "tile",
            "--index-filter", INDEX_FILTER,
            "--reliability-aware",
            "--loss-type", "focal",
            "--focal-gamma", "1.5",
            "--class-weighting",
            "--class-weight-power", "0.5",
            "--standardize-features",
            "--no-use-group-task",
            "--no-weighted-sampler",
            "--phase2-rare-finetune",
            "--phase2-epochs", "12",
            "--phase2-lr", "1e-4",
            "--phase2-sampler-power", "1.0",
            "--phase2-rare-quantile", "0.25",
            "--phase2-rare-boost", "2.0",
            "--batch-size", "48",
            "--output-dir", str(OUTPUT_ROOT / "v4_ssl_reliability_finale"),
        ],
    },
}

VARIANTE_FINALE = "v4_ssl_reliability_finale"

for name, meta in VARIANTES.items():
    suffix = " <- VARIANTE FINALE" if meta.get("is_final", False) else ""
    print(f"{name}: {meta['description']}{suffix}")


## Cellule 6 - (Optionnel) Pré-entraînement SSL

Lance cette cellule seulement si tu veux générer un encodeur SSL dans ce notebook.


In [ ]:
LANCER_SSL = False

if LANCER_SSL:
    ssl_args = [
        "--prepared-npz", str(PREPARED_NPZ),
        "--reliability-aware",
        "--epochs", "30",
        "--batch-size", "64",
        "--lr", "7e-4",
        "--mask-ratio", "0.25",
        "--output-dir", str(OUTPUT_ROOT / "ssl_pretrain"),
    ]
    run_ssl_pretrain(ssl_args)
    print("SSL termine. Recupere ensuite le chemin best_ssl_encoder.pt genere.")
else:
    print("LANCER_SSL=False -> cellule ignoree.")


## Cellule 7 - Exécuter une variante (single run)

Choisis la variante à lancer. Les métriques affichées sont celles du **test**.


In [ ]:
VARIANTE_A_LANCER = VARIANTE_FINALE
SEED = 42

if VARIANTE_A_LANCER not in VARIANTES:
    raise ValueError(f"Variante inconnue: {VARIANTE_A_LANCER}")

meta = VARIANTES[VARIANTE_A_LANCER]
args = list(meta["args"])
args.extend(["--seed", str(SEED)])

if meta.get("requires_ssl", False):
    if not SSL_ENCODER_PATH:
        raise ValueError("La variante finale requiert SSL_ENCODER_PATH.")
    args.extend(["--pretrained-encoder-checkpoint", SSL_ENCODER_PATH])

print("Lancement:", VARIANTE_A_LANCER)
run_train_variant(args)

run_dir = latest_run_dir(Path(meta["args"][meta["args"].index("--output-dir") + 1]))
print("\nDernier run:", run_dir)
if run_dir is not None:
    m = read_test_metrics(run_dir)
    print("\n=== Metriques TEST ===")
    print(f"Exactitude      : {m.get('accuracy', float('nan')):.4f}")
    print(f"Precision macro : {m.get('precision_macro', float('nan')):.4f}")
    print(f"Rappel macro    : {m.get('recall_macro', float('nan')):.4f}")
    print(f"F1 macro        : {m.get('f1_macro', float('nan')):.4f}")
    print(f"F1 pondere      : {m.get('f1_weighted', float('nan')):.4f}")


## Cellule 8 - Multi-seeds + Ensemble (optionnel)

Relance une variante sur plusieurs seeds, puis évalue:
- ensemble uniforme,
- ensemble pondéré par macro-F1 validation.


In [ ]:
LANCER_MULTI_SEEDS = False
SEEDS = [42, 123, 777, 2024]
VARIANTE_MULTI = VARIANTE_FINALE

if LANCER_MULTI_SEEDS:
    if VARIANTE_MULTI not in VARIANTES:
        raise ValueError(f"Variante inconnue: {VARIANTE_MULTI}")

    meta = VARIANTES[VARIANTE_MULTI]
    if meta.get("requires_ssl", False) and not SSL_ENCODER_PATH:
        raise ValueError("Renseigne SSL_ENCODER_PATH pour la variante finale multi-seeds.")

    seed_out_root = OUTPUT_ROOT / f"{VARIANTE_MULTI}_seeds"
    seed_out_root.mkdir(parents=True, exist_ok=True)

    for seed in SEEDS:
        args = list(meta["args"])
        # remplace output-dir pour chaque seed
        out_idx = args.index("--output-dir") + 1
        args[out_idx] = str(seed_out_root / f"s{seed}")
        args.extend(["--seed", str(seed)])

        if meta.get("requires_ssl", False):
            args.extend(["--pretrained-encoder-checkpoint", SSL_ENCODER_PATH])

        print(f"\n=== Run seed={seed} ===")
        run_train_variant(args)

    ckpt_glob = str(seed_out_root / "s*" / "temporal_transformer_*" / "best_model.pt")

    print("\n=== Ensemble uniforme ===")
    run_ensemble_eval([
        "--checkpoint-glob", ckpt_glob,
        "--prepared-npz", str(PREPARED_NPZ),
        "--ensemble-weighting", "uniform",
        "--output-dir", str(seed_out_root / "ensemble_uniform"),
    ])

    print("\n=== Ensemble pondéré (val_macro_f1) ===")
    run_ensemble_eval([
        "--checkpoint-glob", ckpt_glob,
        "--prepared-npz", str(PREPARED_NPZ),
        "--ensemble-weighting", "val_macro_f1",
        "--weight-power", "1.0",
        "--output-dir", str(seed_out_root / "ensemble_weighted"),
    ])
else:
    print("LANCER_MULTI_SEEDS=False -> cellule ignoree.")


## Cellule 9 - Lire les métriques d'un dossier existant

Permet de réafficher rapidement les scores déjà produits.


In [ ]:
DOSSIER_RESULTATS = ""  # ex: outputs_transformer_notebook/v4_ssl_reliability_finale/temporal_transformer_YYYYMMDD_HHMMSS

if DOSSIER_RESULTATS:
    d = Path(DOSSIER_RESULTATS)
    m = read_test_metrics(d)
    print("Dossier:", d)
    print(f"Exactitude      : {m.get('accuracy', float('nan')):.4f}")
    print(f"Precision macro : {m.get('precision_macro', float('nan')):.4f}")
    print(f"Rappel macro    : {m.get('recall_macro', float('nan')):.4f}")
    print(f"F1 macro        : {m.get('f1_macro', float('nan')):.4f}")
    print(f"F1 pondere      : {m.get('f1_weighted', float('nan')):.4f}")
else:
    print("Renseigne DOSSIER_RESULTATS pour relire un run.")
